<a href="https://colab.research.google.com/github/paolazcastillo/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System/blob/main/RNN_DATASET_CATEGORIZATION_OF_LENGTHS_IN_THE_RHESUS_MONKEY_(MACACA_MULATTA)_WITH_A_THREE_CATEGORY_SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **RNN-ready dataset**

# Categorization of Lengths in the Rhesus Monkey (Macaca Mulatta) with a Three-Category System

Paola Castillo

This notebook turns every session in `outputs/` into the sequence dataset a
recurrent model consumes: trial windows on a uniform grid, the epoch timeline,
the split by session, condition-averaged velocity profiles, a time-resolved
decoding baseline, functional PCA of the velocity curves and the tensor export
(`rnn_dataset.npz`).

It is the former section 3 of the across-session EDA notebook and keeps that
number, so its cross-references still hold: "section 1.x" means the
single-session EDA notebook, "section 2.x" the across-session one
(`EDA_SINGLE_SESSION_...ipynb` and `EDA_MULTI_SESSION_...ipynb`). Sessions are
always ordered by the date parsed from their runTag
(`sessROM_<dd-mmm-yyyy>_<HH-MM>`). Run 0.1, 0.2 and then section 3 top to bottom.


## 0. GitHub Repository Connection

In [ ]:
# Remove any clone left by an earlier run of this runtime. Without this,
# `git clone` below fails with "destination path already exists" and the
# notebook silently keeps reading the OLD data.
!rm -rf "/content/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System"

In [ ]:
!git clone https://github.com/paolazcastillo/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System.git

Cloning into 'Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 222 (delta 108), reused 131 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 411.91 KiB | 2.92 MiB/s, done.
Resolving deltas: 100% (108/108), done.


In [ ]:
!ls "/content/Categorization-of-Lengths-in-the-Rhesus-Monkey-Macaca-Mulatta-with-a-Three-Category-System"

centerTask_v8.17  fonts  README.md


## 0.1. Loading sessions from `outputs/`

The task writes one folder per session under `outputs/`, named after the
runTag it stamps everything with
(`sessROM_<dd-mmm-yyyy>_<HH-MM>`, e.g. `sessROM_26-Aug-2026_12-28`). Each
folder holds that session's `trial_data_*.csv`, both trajectory exports,
`session_data_*.csv` and the printed `session_report_*.txt`.

This cell reads that tree directly; every session to be pooled has to live
there.

- **Filtering by date.** Set `SESSION_START_DATE` and `SESSION_END_DATE` to
  anything pandas can parse (`'2026-08-21'`, `'21-Aug-2026'`, a datetime) or
  leave them `None` for no bound. Both ends are inclusive, and a bare end
  date covers that whole day, so passing the same value twice selects one
  day's sessions. The timestamp is parsed from the runTag rather than from
  the `Date` column inside the CSV, which is what lets two sessions recorded
  on the same day still be ordered.

- **What it sets.** `SESSION_INVENTORY` is the table of sessions in the
  window with a path per file kind (`None` where a session predates that
  export). `POOLED_FROM_OUTPUTS` is every trial in the window in one frame,
  tagged with `Session` / `SessionStart` / `SessionIndex`. `SESSION_GLOB` is
  pointed at the same folders; section 3 builds the RNN dataset from
  exactly this window.

The exploratory analyses (per-session and across sessions) are the job of
the two companion EDA notebooks.

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = "outputs"
OUTPUTS_GLOBS = ["outputs", "/content/*/outputs", "../outputs"]
SESSION_START_DATE = None
SESSION_END_DATE = None
RUNTAG_DATE_RE = re.compile(r"_(\d{2}-[A-Za-z]{3}-\d{4})_(\d{2})-(\d{2})$")
SESSION_FILE_KINDS = {
    "trial_data": "trial_data_{tag}.csv",
    "trajectory": "trajectory_{tag}.csv",
    "trajectory_movement": "trajectory_movement_{tag}.csv",
    "session_data": "session_data_{tag}.csv",
    "session_report": "session_report_{tag}.txt",
    "trial_kinematics": "trial_kinematics_{tag}.csv",
    "foil_events": "foil_events_{tag}.csv",
}


def _normalize(td):
    """normalize_trial_schema if the analysis cells have already run, else the
    same canonicalisation inline.

    Duplicated deliberately: this cell is meant to be runnable FIRST, before
    the cells that define the real one, so that the paths it sets are ready
    for them. Keeping the fallback means loading never depends on execution
    order.
    """
    fn = globals().get("normalize_trial_schema")
    if callable(fn):
        return fn(td)
    td = td.copy()
    if "ExecutionTime_s" not in td.columns and "ReactionTime_s" in td.columns:
        td["ExecutionTime_s"] = td["ReactionTime_s"]
        td = td.drop(columns=["ReactionTime_s"])
    if "TotalTime_s" not in td.columns:
        if "ReactionTime_s" in td.columns:
            td["TotalTime_s"] = td["ReactionTime_s"]
        elif {"DecisionTime_s", "ExecutionTime_s"} <= set(td.columns):
            td["TotalTime_s"] = td["DecisionTime_s"] + td["ExecutionTime_s"]
    if "ReactionTime_s" in td.columns:
        td = td.drop(columns=["ReactionTime_s"])
    return td


def resolve_outputs_dir(candidates=OUTPUTS_GLOBS):
    """Find the outputs/ tree the task writes its sessions into.

    Checked in order so the same notebook works from a local checkout and
    from Colab, where the repo lands under /content/<repo>/. Returns None
    rather than raising, so the notebook still loads without it.
    """
    import glob as _g
    for cand in candidates:
        for hit in sorted(_g.glob(cand)):
            if Path(hit).is_dir():
                return Path(hit)
    return None


def session_timestamp(run_tag):
    """Session start time parsed out of the runTag.

    CenterOutTask.m builds it as datestr(now, 'dd-mmm-yyyy_HH-MM'), so
    'sessROM_26-Aug-2026_12-28' carries the real start instant. Parsing the
    tag rather than the Date column inside the CSV is what lets two sessions
    recorded on the SAME day still be ordered, and what makes the date filter
    below work without opening a single file.
    """
    m = RUNTAG_DATE_RE.search(str(run_tag))
    if not m:
        return pd.NaT
    return pd.to_datetime(f"{m.group(1)} {m.group(2)}:{m.group(3)}",
                          format="%d-%b-%Y %H:%M", errors="coerce")


def discover_sessions(outputs_dir=None, start_date=None, end_date=None, verbose=True):
    """Inventory of every session folder under outputs/, filtered by date.

    One row per session, one column per file kind, holding the path when that
    file exists and None when it does not -- older sessions carry no
    foil_events or trial_kinematics, and nothing here should have to care.

    start_date / end_date are inclusive and accept anything pandas can parse
    ('2026-08-21', '21-Aug-2026', a datetime). end_date given as a bare date
    covers that whole day, so passing the same value for both selects one
    day's sessions rather than only those recorded at exactly midnight.
    """
    outputs_dir = Path(outputs_dir) if outputs_dir else resolve_outputs_dir()
    if outputs_dir is None or not outputs_dir.exists():
        print(f"No outputs/ directory found (looked in {OUTPUTS_GLOBS}). "
              f"Set OUTPUTS_DIR to its path and re-run.")
        return pd.DataFrame()
    rows = []
    for d in sorted(p for p in outputs_dir.iterdir() if p.is_dir()):
        tag = d.name
        rec = {"Session": tag, "SessionStart": session_timestamp(tag), "folder": str(d)}
        for kind, pattern in SESSION_FILE_KINDS.items():
            f = d / pattern.format(tag=tag)
            rec[kind] = str(f) if f.exists() else None
        if rec["trial_data"] is None:
            hits = sorted(d.glob("trial_data_*.csv"))
            rec["trial_data"] = str(hits[0]) if hits else None
        rows.append(rec)
    inv = pd.DataFrame(rows)
    if inv.empty:
        print(f"{outputs_dir} has no session sub-folders.")
        return inv
    n_all = len(inv)
    if start_date is not None:
        lo = pd.to_datetime(start_date)
        inv = inv[inv["SessionStart"].notna() & (inv["SessionStart"] >= lo)]
    if end_date is not None:
        hi = pd.to_datetime(end_date)
        if hi == hi.normalize():
            hi = hi + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
        inv = inv[inv["SessionStart"].notna() & (inv["SessionStart"] <= hi)]
    inv = inv.sort_values("SessionStart").reset_index(drop=True)
    inv.insert(0, "index", np.arange(1, len(inv) + 1))
    if verbose:
        window = ""
        if start_date is not None or end_date is not None:
            window = f"  [filter: {start_date or 'earliest'} .. {end_date or 'latest'}]"
        print(f"{outputs_dir}: {len(inv)}/{n_all} sessions{window}\n")
        show = inv[["index", "Session", "SessionStart"]].copy()
        show["files"] = [sum(r[k] is not None for k in SESSION_FILE_KINDS)
                         for _, r in inv.iterrows()]
        print(show.to_string(index=False))
        missing = [k for k in SESSION_FILE_KINDS if inv[k].isna().all()]
        if missing:
            print(f"\n  No session in this window has: {missing} "
                  f"(expected for sessions recorded before those exports existed).")
    return inv


def load_sessions_from_outputs(outputs_dir=None, start_date=None, end_date=None,
                               min_trials=20, verbose=True):
    """Pool the trial tables of every session in the date window into one frame.

    Each file goes through normalize_trial_schema, so a window may freely mix
    engine generations. Session identity and start time are carried on every
    row, because the across-session models need session as a grouping factor.
    """
    inv = discover_sessions(outputs_dir, start_date, end_date, verbose=verbose)
    if inv.empty:
        return pd.DataFrame(), inv
    frames, skipped = [], []
    for _, r in inv.iterrows():
        if not r["trial_data"]:
            skipped.append((r["Session"], "no trial_data_*.csv")); continue
        try:
            td = _normalize(pd.read_csv(r["trial_data"]))
        except Exception as exc:
            skipped.append((r["Session"], f"unreadable: {exc}")); continue
        if len(td) < min_trials:
            skipped.append((r["Session"], f"only {len(td)} trials")); continue
        td["Session"] = r["Session"]
        td["SessionStart"] = r["SessionStart"]
        td["SessionIndex"] = r["index"]
        td["SourceFile"] = Path(r["trial_data"]).name
        frames.append(td)
    if not frames:
        print("No usable trial tables in this window.")
        for sid, why in skipped: print(f"  skipped {sid}: {why}")
        return pd.DataFrame(), inv
    pooled = pd.concat(frames, ignore_index=True, sort=False)
    if verbose:
        print(f"\nPooled {pooled['Session'].nunique()} sessions, {len(pooled)} trials.")
        for sid, why in skipped: print(f"  skipped {sid}: {why}")
        per = (pooled.groupby(["SessionIndex", "Session"])
               .agg(trials=("IsCorrect", "size"), accuracy=("IsCorrect", "mean")).reset_index())
        print("\n" + per.round(4).to_string(index=False))
    return pooled, inv


REQUIRED_TRIAL_FIELDS = ["NumCategories", "SessionMode"]


def check_session_fields(inv, fields=REQUIRED_TRIAL_FIELDS):
    """Warn when a trial_data file lacks a field the analyses read, and say
    WHICH file was read: a NaN in NumCategories / SessionMode for a session
    that has them in the repository means the notebook is reading a stale
    copy (typically a Colab runtime whose old clone was not removed)."""
    if inv is None or inv.empty:
        return
    missing = []
    for _, r in inv.iterrows():
        if not r.get("trial_data"):
            continue
        with open(r["trial_data"]) as fh:
            header = fh.readline().strip().split(",")
        lack = [f for f in fields if f not in header]
        if lack:
            missing.append((r["Session"], lack, r["trial_data"]))
    if missing:
        print(f"\nWARNING: {len(missing)} session file(s) lack {fields}:")
        for tag, lack, path in missing:
            print(f"  {tag}: missing {lack}  <- {path}")
        print("  If the repository has these columns, the files being read are stale: "
              "re-run the clone cell in section 0 (it now removes the old clone) "
              "or pull the latest outputs/.")


SESSION_INVENTORY = discover_sessions(OUTPUTS_DIR if Path(OUTPUTS_DIR).exists() else None,
                                      SESSION_START_DATE, SESSION_END_DATE)
check_session_fields(SESSION_INVENTORY)
POOLED_FROM_OUTPUTS, _INV = load_sessions_from_outputs(
    OUTPUTS_DIR if Path(OUTPUTS_DIR).exists() else None,
    SESSION_START_DATE, SESSION_END_DATE, verbose=True)

if not SESSION_INVENTORY.empty:
    _resolved = resolve_outputs_dir() if not Path(OUTPUTS_DIR).exists() else Path(OUTPUTS_DIR)
    SESSION_GLOB = str(Path(_resolved) / "*" / "trial_data_*.csv")
    print(f"\nSESSION_GLOB set to {SESSION_GLOB!r} -- section 2 will pool the same sessions.")


## 0.2. Shared definitions

Section 3 reuses the signal-processing pipeline of section 1.1 of the
single-session notebook (`TrajectoryProcessor`, `normalize_trial_schema`, the
epoch codes, `FIGURE_DPI` and the figure style). The cell below is a verbatim
copy of what it needs; it only defines functions and constants. The pipeline
itself is used by the non-causal comparison in 3.5 only, the default
resampler in 3.1 is causal. If the pipeline changes in the single-session
notebook, update the copy here.


In [ ]:
"""Signal-processing pipeline, copied from section 1.1 of the single-session
notebook (constants, font/figure style, Hampel -> Kalman+RTS -> PCHIP ->
Butterworth, TrajectoryDataset, normalize_trial_schema, TrialInfoTable).

Only the pieces the across-session cells read are kept: the per-trial figure
drawer and run() stay in the single-session notebook. Keep the two copies in
sync when the pipeline changes.
"""

import glob
import zipfile
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator
from scipy.signal import lfilter, savgol_filter
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import MultipleLocator
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize

GRID_DT_S = 0.008
CUTOFF_HZ = 6.0
ACCEL_PEAK_PERCENTILE = 99.0
SAVGOL_WINDOW = 11
SAVGOL_POLYORDER = 3
MOVE_SPEED_FRAC = 0.05
PIXEL_PITCH_MM = 0.3108
SHOW_INLINE = True
OUTPUT_DIR = "figures"
FIX_TIME_AXIS = True
FIX_POS_AXIS = True
TIME_TICK_STEP_MS = 300
AXIS_MARGIN = 0.03
SCREEN_HALF_EXTENT_PX = 750

FIX_ACCEL_SCALE = True
ACCEL_SCALE_PERCENTILE = 99.0

DECISION_EPOCH = 6
MOVEMENT_EPOCH = 7
TARGET_HOLD_EPOCH = 8
MOVE_EPOCHS = (DECISION_EPOCH, MOVEMENT_EPOCH)

_EPOCH_NAME_TO_CODE = {
    "REACTION": DECISION_EPOCH, "DECISION": DECISION_EPOCH,
    "DECISION_TIME": DECISION_EPOCH, "DECISIONTIME": DECISION_EPOCH,
    "MOVEMENT": MOVEMENT_EPOCH, "MOVE": MOVEMENT_EPOCH, "EXECUTION": MOVEMENT_EPOCH,
    "TARGET_HOLD": TARGET_HOLD_EPOCH, "TARGETHOLD": TARGET_HOLD_EPOCH, "HOLD": TARGET_HOLD_EPOCH,
}


def _epoch_to_code(v):
    """Numeric TaskEpoch code from a numeric OR string Epoch cell (NaN if unknown)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    try:
        return int(v)
    except (ValueError, TypeError):
        return _EPOCH_NAME_TO_CODE.get(str(v).strip().upper(), np.nan)

CENTER_TO_TARGET_DIST_PX = 320
TARGET_DIST_SCALE = 1.27
TARGET_DIAMETER_PX = 180
CENTER_DIAMETER_PX = 200
TARGET_DIST_PX = CENTER_TO_TARGET_DIST_PX * TARGET_DIST_SCALE
TARGET_RADIUS_PX = TARGET_DIAMETER_PX / 2.0
CENTER_RADIUS_PX = CENTER_DIAMETER_PX / 2.0
SCREEN_WIDTH_PX = 1920
SCREEN_HEIGHT_PX = 1080
USE_SCREEN_CENTER = True
SCREEN_VIEW_FULL = True

CM_PER_PX = PIXEL_PITCH_MM / 10.0

FONT_CANDIDATES = ["fonts/harding.ttf", "fonts/Harding.ttf"]
FONT_GLOBS = ["/content/*/fonts/harding.ttf", "/content/*/fonts/Harding.ttf"]


def _resolve_font():
    for cand in FONT_CANDIDATES:
        if Path(cand).exists():
            return cand
    for pattern in FONT_GLOBS:
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits[0]
    return None


def _register_font(path):
    try:
        if path and Path(path).exists():
            fm.fontManager.addfont(path)
            plt.rcParams["font.family"] = fm.FontProperties(fname=path).get_name()
            return True
    except Exception:
        pass
    return False


plt.rcParams["axes.unicode_minus"] = False
FONT_PATH = _resolve_font()
if _register_font(FONT_PATH):
    print(f"Using font: {FONT_PATH}")
else:
    print("Harding font not found (looked in fonts/harding.ttf and /content/*/fonts/); using default font.")


FIGURE_DPI = 300
BASE_FONT_SIZE = 10

# Figure ink, storytelling-with-data style: everything that is context is grey,
# colour is spent only on the mark the reader should look at. Category colours
# stay the rig's (ColorCategoryMap.m), because a figure and the screen must agree.
INK = "#333333"          # text, emphasised lines
GRAY = "#8c8c8c"         # context marks, error bars, reference lines
GRAY_LIGHT = "#d9d9d9"   # de-emphasised fills, bands, chance lines
ACCENT = "#31688e"       # the one series the panel is about
ACCENT_2 = "#d44842"     # a second, warm accent (incorrect, a contrasting series)


def apply_figure_style(dpi=FIGURE_DPI, base=BASE_FONT_SIZE):
    """One font, one export resolution and one ink for every figure.

    rcParams are global to the kernel, so calling this once here -- before
    anything is drawn -- is what makes the whole notebook consistent instead
    of each cell setting its own sizes. savefig.dpi is set as well as the
    explicit dpi= arguments, so even a figure saved without one comes out at
    the same resolution. font.family is left alone: it was just set from
    harding.ttf above, and overriding it here would undo that.

    The chrome follows the storytelling-with-data rules: no top/right spines,
    grey hairline axes and ticks, no grid, frameless legends, left-aligned
    titles, so the data ink is the darkest thing on the page.
    """
    plt.rcParams.update({
        "figure.dpi": 110,
        "savefig.dpi": dpi,
        "savefig.bbox": "tight",
        "font.size": base,
        "axes.titlesize": base + 1,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "figure.titlesize": base + 2,
        "axes.unicode_minus": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": GRAY,
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK,
        "axes.titlecolor": INK,
        "axes.titlelocation": "left",
        "axes.grid": False,
        "xtick.color": GRAY,
        "ytick.color": GRAY,
        "xtick.labelcolor": INK,
        "ytick.labelcolor": INK,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "text.color": INK,
        "legend.frameon": False,
        "lines.linewidth": 1.6,
        "lines.markersize": 5,
        "errorbar.capsize": 0,
    })


apply_figure_style()


class HampelScreen:
    MAD_SCALE = 1.4826

    def __init__(self, half_window=3, n_sigma=3.0):
        self.half_window = max(1, int(round(half_window)))
        self.n_sigma = float(n_sigma)

    def detect(self, x):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        if x.shape[0] == 1 and x.shape[1] > 1:
            x = x.T
        n, d = x.shape
        cleaned = x.copy()
        median_env = x.copy()
        threshold_env = np.zeros_like(x)
        mask = np.zeros_like(x, dtype=bool)
        if n < 3:
            return cleaned, mask, median_env, threshold_env
        h = self.half_window
        for col in range(d):
            source = x[:, col]
            for i in range(n):
                lo = max(0, i - h)
                hi = min(n, i + h + 1)
                window = source[lo:hi]
                med = np.median(window)
                mad = self.MAD_SCALE * np.median(np.abs(window - med))
                thr = self.n_sigma * mad
                median_env[i, col] = med
                threshold_env[i, col] = thr
                if mad > 0 and abs(source[i] - med) > thr:
                    cleaned[i, col] = med
                    mask[i, col] = True
        return cleaned, mask, median_env, threshold_env


class ButterworthLowpass:
    N_FACT = 6

    def __init__(self, cutoff_hz=20.0):
        self.cutoff_hz = float(cutoff_hz)

    def _design(self, fs):
        k = np.tan(np.pi * self.cutoff_hz / fs)
        norm = 1.0 / (1.0 + np.sqrt(2.0) * k + k ** 2)
        b = np.array([k ** 2, 2.0 * k ** 2, k ** 2]) * norm
        a = np.array([1.0, 2.0 * (k ** 2 - 1.0) * norm, (1.0 - np.sqrt(2.0) * k + k ** 2) * norm])
        return b, a

    @staticmethod
    def _steady_state(b, a):
        companion = np.array([[-a[1], 1.0], [-a[2], 0.0]])
        rhs = np.array([b[1] - a[1] * b[0], b[2] - a[2] * b[0]])
        return np.linalg.solve(np.eye(2) - companion, rhs)

    def __call__(self, x, fs):
        x = np.atleast_2d(np.asarray(x, dtype=float))
        if x.shape[0] == 1 and x.shape[1] > 1:
            x = x.T
        n, d = x.shape
        y = x.copy()
        if n == 0 or fs <= 0 or self.cutoff_hz <= 0 or self.cutoff_hz >= fs / 2.0:
            return y, False
        if n <= self.N_FACT:
            return y, False
        b, a = self._design(fs)
        zi = self._steady_state(b, a)
        nf = self.N_FACT
        for col in range(d):
            series = x[:, col]
            pre = 2.0 * series[0] - series[nf:0:-1]
            post = 2.0 * series[-1] - series[-2:-nf - 2:-1]
            ext = np.concatenate([pre, series, post])
            ext, _ = lfilter(b, a, ext, zi=zi * ext[0])
            ext = ext[::-1]
            ext, _ = lfilter(b, a, ext, zi=zi * ext[0])
            ext = ext[::-1]
            y[:, col] = ext[nf:-nf]
        return y, True


class KalmanTrajectorySmoother:
    def __init__(self, meas_sigma_px=2.0, jerk_sigma_px_s3=1e5, gate_sigma=np.inf):
        self.meas_sigma_px = float(meas_sigma_px)
        self.jerk_sigma_px_s3 = float(jerk_sigma_px_s3)
        self.gate_sigma = float(gate_sigma)

    def __call__(self, t, xy):
        t = np.asarray(t, dtype=float).ravel()
        xy = np.atleast_2d(np.asarray(xy, dtype=float))
        if xy.shape[0] == 1 and xy.shape[1] > 1:
            xy = xy.T
        n = t.size
        smoothed = xy.copy()
        n_gated = 0
        if n < 3 or xy.shape[0] != n:
            return smoothed, n_gated
        r = self.meas_sigma_px ** 2
        q = self.jerk_sigma_px_s3 ** 2
        dt_all = np.diff(t)
        acc_std0 = self.jerk_sigma_px_s3 * 20.0 * np.median(dt_all)
        gate2 = self.gate_sigma ** 2
        for col in range(xy.shape[1]):
            z = xy[:, col]
            xf = np.zeros((3, n))
            pf = np.zeros((3, 3, n))
            xp = np.zeros((3, n))
            pp = np.zeros((3, 3, n))
            fk = np.zeros((3, 3, n))
            dt0 = dt_all[0]
            xp[:, 0] = [z[0], (z[1] - z[0]) / dt0, 0.0]
            pp[:, :, 0] = np.diag([r, 2.0 * r / dt0 ** 2, acc_std0 ** 2])
            fk[:, :, 0] = np.eye(3)
            for k in range(n):
                if k > 0:
                    dt = dt_all[k - 1]
                    f = np.array([[1.0, dt, dt ** 2 / 2.0],
                                  [0.0, 1.0, dt],
                                  [0.0, 0.0, 1.0]])
                    qm = q * np.array([[dt ** 5 / 20.0, dt ** 4 / 8.0, dt ** 3 / 6.0],
                                       [dt ** 4 / 8.0, dt ** 3 / 3.0, dt ** 2 / 2.0],
                                       [dt ** 3 / 6.0, dt ** 2 / 2.0, dt]])
                    fk[:, :, k] = f
                    xp[:, k] = f @ xf[:, k - 1]
                    pp[:, :, k] = f @ pf[:, :, k - 1] @ f.T + qm
                innov = z[k] - xp[0, k]
                s = pp[0, 0, k] + r
                if innov ** 2 <= gate2 * s:
                    gain = pp[:, 0, k] / s
                    xf[:, k] = xp[:, k] + gain * innov
                    pf[:, :, k] = pp[:, :, k] - np.outer(gain, pp[0, :, k])
                    pf[:, :, k] = (pf[:, :, k] + pf[:, :, k].T) / 2.0
                else:
                    xf[:, k] = xp[:, k]
                    pf[:, :, k] = pp[:, :, k]
                    n_gated += 1
            xs = xf.copy()
            for k in range(n - 2, -1, -1):
                f = fk[:, :, k + 1]
                c = pf[:, :, k] @ f.T @ np.linalg.inv(pp[:, :, k + 1])
                xs[:, k] = xf[:, k] + c @ (xs[:, k + 1] - xp[:, k + 1])
            smoothed[:, col] = xs[0, :]
        return smoothed, n_gated


@dataclass
class TrialResult:
    raw_time_ms: np.ndarray
    raw_xy: np.ndarray
    grid_time_ms: np.ndarray
    grid_xy: np.ndarray
    outlier_mask: np.ndarray
    hampel_median: np.ndarray
    hampel_threshold: np.ndarray
    n_replaced: int
    butter_applied: bool
    under_resolved: bool
    peak_vel_cm: float
    mean_vel_cm: float
    median_vel_cm: float
    peak_accel_cm: float
    mean_accel_cm: float
    median_accel_cm: float
    vel_time_ms: np.ndarray
    vel_cm: np.ndarray
    accel_cm: np.ndarray
    move_onset_ms: float = float("nan")
    move_offset_ms: float = float("nan")
    move_takeoff_ms: float = float("nan")


class TrajectoryProcessor:
    def __init__(self, grid_dt_s=0.008, cutoff_hz=CUTOFF_HZ, cm_per_px=CM_PER_PX,
                min_move_samples=5, min_move_dur_s=None,
                hampel_half_window=3, hampel_n_sigma=3.0,
                meas_sigma_px=2.0, jerk_sigma_px_s3=1e5, gate_sigma=np.inf):
        self.grid_dt_s = grid_dt_s
        self.cutoff_hz = cutoff_hz
        self.cm_per_px = cm_per_px
        self.min_move_samples = min_move_samples
        self.min_move_dur_s = min_move_dur_s
        self.hampel = HampelScreen(hampel_half_window, hampel_n_sigma)
        self.butter = ButterworthLowpass(cutoff_hz)
        self.kalman = KalmanTrajectorySmoother(meas_sigma_px, jerk_sigma_px_s3, gate_sigma)

    @staticmethod
    def _savgol_accel(vel_cm, dt_s, window, poly):
        n = int(vel_cm.size)
        if n == 0:
            return np.array([])
        if n == 1:
            return np.array([0.0])
        if n < 5:
            return np.abs(np.gradient(vel_cm, dt_s))
        w = min(int(window), n if n % 2 == 1 else n - 1)
        if w % 2 == 0:
            w -= 1
        w = max(w, 5)
        p = min(int(poly), w - 1)
        return np.abs(savgol_filter(vel_cm, w, p, deriv=1, delta=dt_s))

    def _build_grid(self, t):
        total = t[-1] - t[0]
        if total < self.grid_dt_s:
            return np.array([t[0], t[-1]])
        grid = np.arange(t[0], t[-1] + 1e-12, self.grid_dt_s)
        if t[-1] - grid[-1] > self.grid_dt_s / 4.0:
            grid = np.append(grid, t[-1])
        else:
            grid[-1] = t[-1]
        return grid

    def _kinematics(self, grid_time_ms, grid_xy, under_resolved, move_window_ms0=None):
        nan = float("nan")
        empty = np.array([])
        c = self.cm_per_px
        if under_resolved:
            return nan, nan, nan, nan, empty, empty, empty, nan, nan, nan
        t = grid_time_ms / 1000.0
        gdt = np.diff(t)
        if gdt.size < 1 or np.any(gdt <= 0):
            return nan, nan, nan, nan, empty, empty, empty, nan, nan, nan
        vel = np.hypot(np.diff(grid_xy[:, 0]), np.diff(grid_xy[:, 1])) / gdt
        tv_ms = (t[:-1] + gdt / 2.0) * 1000.0
        vel_cm = vel * c
        dt_s = float(np.median(gdt))
        accel_cm = self._savgol_accel(vel_cm, dt_s, SAVGOL_WINDOW, SAVGOL_POLYORDER)
        peak = float(np.max(vel))
        if peak < 1e-6:
            return nan, nan, nan, nan, tv_ms, vel_cm, accel_cm, nan, nan, nan
        if move_window_ms0 is not None:
            mmask = (tv_ms >= move_window_ms0[0]) & (tv_ms <= move_window_ms0[1])
            if not mmask.any():
                mmask = np.ones(tv_ms.shape, dtype=bool)
        else:
            mmask = np.ones(tv_ms.shape, dtype=bool)
        takeoff_ms = nan
        if vel_cm.size and mmask.any():
            mov_idx = np.where(mmask)[0]
            pk_idx = int(mov_idx[np.argmax(vel_cm[mov_idx])])
            thr = MOVE_SPEED_FRAC * float(vel_cm[pk_idx])
            lo_idx = pk_idx
            while lo_idx > 0 and vel_cm[lo_idx - 1] >= thr:
                lo_idx -= 1
            takeoff_ms = float(tv_ms[lo_idx])
        mean = float(np.mean(vel[mmask]))
        median_vel = float(np.median(vel[mmask]))
        peak_acc = nan
        mean_acc = nan
        median_acc = nan
        finite_full = accel_cm[np.isfinite(accel_cm)] if accel_cm.size else accel_cm
        if finite_full.size:
            peak_acc = float(np.percentile(finite_full, ACCEL_PEAK_PERCENTILE))
        acc_mov = accel_cm[mmask] if accel_cm.size else accel_cm
        finite_mov = acc_mov[np.isfinite(acc_mov)] if acc_mov.size else acc_mov
        if finite_mov.size:
            mean_acc = float(np.mean(finite_mov))
            median_acc = float(np.median(finite_mov))
        return (peak * c, mean * c, peak_acc, mean_acc, tv_ms, vel_cm, accel_cm,
                takeoff_ms, median_vel * c, median_acc)

    def process(self, time_ms, x, y, move_window_ms=None, hold_start_ms=None):
        t = np.asarray(time_ms, dtype=float) / 1000.0
        order = np.argsort(t)
        t = t[order]
        xy = np.column_stack([np.asarray(x, dtype=float)[order],
                            np.asarray(y, dtype=float)[order]])
        t_unique, keep = np.unique(t, return_index=True)
        t = t_unique
        xy = xy[keep]
        n = t.size
        under_resolved = n < self.min_move_samples
        if self.min_move_dur_s is not None and n >= 2:
            under_resolved = under_resolved or (t[-1] - t[0]) < self.min_move_dur_s
        screened, outlier_mask, median_env, threshold_env = self.hampel.detect(xy)
        n_replaced = int(outlier_mask.sum())
        stage1, _ = self.kalman(t, screened)
        if n < 2:
            grid = t.copy()
            grid_xy = stage1.copy()
            applied = False
        else:
            grid = self._build_grid(t)
            grid_xy = np.column_stack([
                PchipInterpolator(t, stage1[:, 0])(grid),
                PchipInterpolator(t, stage1[:, 1])(grid),
            ])
            grid_fs = 1.0 / self.grid_dt_s
            grid_xy, applied = self.butter(grid_xy, grid_fs)
        grid_time_ms = (grid - t[0]) * 1000.0
        t0_ms = t[0] * 1000.0
        move_window_ms0 = None
        move_onset_ms = float("nan")
        if move_window_ms is not None:
            move_window_ms0 = (move_window_ms[0] - t0_ms, move_window_ms[1] - t0_ms)
            move_onset_ms = move_window_ms0[0]
        move_offset_ms = (hold_start_ms - t0_ms) if hold_start_ms is not None else float("nan")
        pv, mv, pa, ma, vel_time_ms, vel_cm, accel_cm, move_takeoff_ms, mdv, mda = \
            self._kinematics(grid_time_ms, grid_xy, under_resolved, move_window_ms0)
        return TrialResult(
            raw_time_ms=(t - t[0]) * 1000.0,
            raw_xy=xy,
            grid_time_ms=grid_time_ms,
            grid_xy=grid_xy,
            outlier_mask=outlier_mask,
            hampel_median=median_env,
            hampel_threshold=threshold_env,
            n_replaced=n_replaced,
            butter_applied=applied,
            under_resolved=under_resolved,
            peak_vel_cm=pv,
            mean_vel_cm=mv,
            median_vel_cm=mdv,
            peak_accel_cm=pa,
            mean_accel_cm=ma,
            median_accel_cm=mda,
            vel_time_ms=vel_time_ms,
            vel_cm=vel_cm,
            accel_cm=accel_cm,
            move_onset_ms=move_onset_ms,
            move_offset_ms=move_offset_ms,
            move_takeoff_ms=move_takeoff_ms,
        )


@dataclass
class Trial:
    date: str
    block: int
    trial: int
    attempt: int
    time_ms: np.ndarray
    move_time_ms: np.ndarray
    x: np.ndarray
    y: np.ndarray
    fs_hz: float
    move_window_ms: tuple = None
    hold_start_ms: float = None
    hold_window_ms: tuple = None
    hold_max_speed_cm: float = None
    hold_excursion_px: float = None

    @property
    def n_samples(self):
        return int(self.x.size)

    @property
    def label(self):
        return f"b{self.block:02d}_t{self.trial:02d}_a{self.attempt:02d}"


class TrajectoryDataset:
    GROUP_KEYS = ["Block", "TrialNumInBlock", "Attempt"]

    def __init__(self, csv_path, move_epochs=None):
        """
        move_epochs: None = no epoch filtering (use every row in csv_path
        as-is). Otherwise a single TaskEpoch code (e.g. MOVEMENT_EPOCH) or
        an iterable of codes (e.g. MOVE_EPOCHS = (DECISION_EPOCH,
        MOVEMENT_EPOCH, TARGET_HOLD_EPOCH)) -- a row is kept when its Epoch
        matches ANY of them. This mirrors CenterOutTask.m's kinematicsEpochs
        / the ismember(...) filter TrialKinematics.m and
        SaveMovementTrajectory.m now use on the MATLAB side, applied here
        explicitly instead of trusting the CSV to already be restricted the
        way we expect. Note this can only ever KEEP rows that exist in
        csv_path already -- if the session was recorded before
        kinematicsEpochs included a given epoch, that epoch's rows were
        never exported and this filter finds nothing to add for it (see the
        module-level comment above MOVE_EPOCHS).
        """
        self.path = Path(csv_path)
        self.frame = pd.read_csv(self.path)
        if "Epoch" in self.frame.columns and not pd.api.types.is_numeric_dtype(self.frame["Epoch"]):
            mapped = self.frame["Epoch"].map(_epoch_to_code)
            n_bad = int(mapped.isna().sum())
            if n_bad:
                unknown = sorted(set(self.frame.loc[mapped.isna(), "Epoch"].astype(str)))[:6]
                print(f"WARNING: {n_bad} rows have unrecognized Epoch names {unknown}; "
                    f"left unmatched -- add them to _EPOCH_NAME_TO_CODE if needed.")
            self.frame["Epoch"] = mapped
            print(f"Epoch column was text; normalized to numeric codes "
                f"{sorted(set(self.frame['Epoch'].dropna().astype(int)))}.")
        if move_epochs is not None:
            if "Epoch" not in self.frame.columns:
                print(f"WARNING: '{self.path.name}' has no Epoch column -- "
                    f"cannot filter to move_epochs={move_epochs}; using every row as-is.")
            else:
                epochs = np.atleast_1d(move_epochs)
                before = len(self.frame)
                self.frame = self.frame[self.frame["Epoch"].isin(epochs)]
                print(f"Epoch filter {tuple(epochs)}: kept {len(self.frame)}/{before} rows of '{self.path.name}'.")

    @staticmethod
    def estimate_sampling_rate(time_ms):
        time_ms = np.asarray(time_ms, dtype=float)
        if time_ms.size < 2:
            return float("nan")
        steps = np.diff(np.sort(time_ms))
        steps = steps[steps > 0]
        if steps.size == 0:
            return float("nan")
        return 1000.0 / float(np.mean(steps))

    @staticmethod
    def _move_time_ms(group):
        if "MoveTime_ms" in group.columns:
            return group["MoveTime_ms"].to_numpy(dtype=float)
        tms = group["Time_ms"].to_numpy(dtype=float)
        return tms - float(tms.min()) if tms.size else tms

    def trials(self):
        for (block, trial, attempt), group in self.frame.groupby(self.GROUP_KEYS, sort=True):
            group = group.sort_values("Time_ms")
            move_window_ms = None
            hold_start_ms = None
            hold_window_ms = None
            hold_max_speed_cm = None
            hold_excursion_px = None
            if "Epoch" in group.columns:
                ep = group["Epoch"].to_numpy()
                tms = group["Time_ms"].to_numpy(dtype=float)
                xs_ = group["X_px"].to_numpy(dtype=float)
                ys_ = group["Y_px"].to_numpy(dtype=float)
                mv = tms[ep == MOVEMENT_EPOCH]
                if mv.size:
                    move_window_ms = (float(mv.min()), float(mv.max()))
                hmask = ep == TARGET_HOLD_EPOCH
                hold = tms[hmask]
                if hold.size:
                    hold_start_ms = float(hold.min())
                if hold.size >= 2:
                    hx, hy = xs_[hmask], ys_[hmask]
                    hold_window_ms = (float(hold.min()), float(hold.max()))
                    hdt = np.diff(hold) / 1000.0
                    good = hdt > 0
                    if good.any():
                        seg = np.hypot(np.diff(hx), np.diff(hy))[good] / hdt[good] * CM_PER_PX
                        hold_max_speed_cm = float(seg.max()) if seg.size else None
                    hold_excursion_px = float(np.max(np.hypot(hx - hx[0], hy - hy[0])))
            yield Trial(
                date=str(group["Date"].iloc[0]),
                block=int(block),
                trial=int(trial),
                attempt=int(attempt),
                time_ms=group["Time_ms"].to_numpy(dtype=float),
                move_time_ms=self._move_time_ms(group),
                x=group["X_px"].to_numpy(dtype=float),
                y=group["Y_px"].to_numpy(dtype=float),
                fs_hz=self.estimate_sampling_rate(group["Time_ms"].to_numpy()),
                move_window_ms=move_window_ms,
                hold_start_ms=hold_start_ms,
                hold_window_ms=hold_window_ms,
                hold_max_speed_cm=hold_max_speed_cm,
                hold_excursion_px=hold_excursion_px,
            )

    def global_limits(self, margin=0.03):
        t_max = 0.0
        p_min, p_max = np.inf, -np.inf
        for tr in self.trials():
            t_max = max(t_max, float(tr.move_time_ms.max()))
            p_min = min(p_min, float(tr.x.min()), float(tr.y.min()))
            p_max = max(p_max, float(tr.x.max()), float(tr.y.max()))
        pad = (p_max - p_min) * margin
        return (0.0, t_max), (p_min - pad, p_max + pad)

    def global_space_limits(self, margin=0.05):
        x_min, x_max = np.inf, -np.inf
        y_min, y_max = np.inf, -np.inf
        for tr in self.trials():
            x_min = min(x_min, float(tr.x.min()))
            x_max = max(x_max, float(tr.x.max()))
            y_min = min(y_min, float(tr.y.min()))
            y_max = max(y_max, float(tr.y.max()))
        px = (x_max - x_min) * margin
        py = (y_max - y_min) * margin
        return (x_min - px, x_max + px), (y_min - py, y_max + py)


def normalize_trial_schema(td):
    """Canonicalize the timing columns of a trial_data_*.csv across engine
    generations, so old and new sessions can be analysed by the same code.

    Four layouts exist in the wild. BOTH header spellings of the total are
    accepted on input, and the older ReactionTime_s header names a DIFFERENT
    interval in two of them -- which is the whole reason this function exists:

      gen A (v2_2 / pre-rename)   DecisionTime_s, ReactionTime_s, TotalTime_s
                                  ReactionTime_s = leave-center -> reach-target
                                  TotalTime_s    = decision + execution
      gen B (interim)             DecisionTime_s, ExecutionTime_s, TotalTime_s
      gen C (v8.19)               DecisionTime_s, ExecutionTime_s, ReactionTime_s
                                  ReactionTime_s = decision + execution (TOTAL)
      gen D (v8.20 onward)        DecisionTime_s, ExecutionTime_s, TotalTime_s
                                  TotalTime_s    = decision + execution (TOTAL)

    gen A is separated from the rest by whether ExecutionTime_s is present,
    which is the one signal that distinguishes them:
      present -> a ReactionTime_s column, if any, is the TOTAL   (gen C)
      absent  -> ReactionTime_s is the EXECUTION time            (gen A)

    Canonical output, always these three whatever came in:
      DecisionTime_s   target-onset  -> leave-center
      ExecutionTime_s  leave-center  -> reach-target
      TotalTime_s      DecisionTime_s + ExecutionTime_s

    ReactionTime_s is dropped after being folded into TotalTime_s: keeping
    both would put two identical columns into the correlation heatmap and
    the PCA, where an exact duplicate is not a second measurement.
    """
    td = td.copy()
    if "ExecutionTime_s" not in td.columns and "ReactionTime_s" in td.columns:
        td["ExecutionTime_s"] = td["ReactionTime_s"]
        td = td.drop(columns=["ReactionTime_s"])
    if "TotalTime_s" not in td.columns:
        if "ReactionTime_s" in td.columns:
            td["TotalTime_s"] = td["ReactionTime_s"]
        elif {"DecisionTime_s", "ExecutionTime_s"} <= set(td.columns):
            td["TotalTime_s"] = td["DecisionTime_s"] + td["ExecutionTime_s"]
    if "ReactionTime_s" in td.columns:
        td = td.drop(columns=["ReactionTime_s"])
    return td


class TrialInfoTable:
    """Per-trial lookups keyed on (Block, TrialNumInBlock, Attempt).

    bar_size_deg / direction_correct / is_correct / error_type always come
    from trial_data_<runTag>.csv.

    move_samples (NumMovementSamples) moved OUT of trial_data_<runTag>.csv
    into its own companion file, trial_kinematics_<runTag>.csv, on the
    MATLAB side (same runTag, same Block+TrialNumInBlock+Attempt key --
    see CenterOutTask.m). To stay usable on BOTH older sessions (column
    still sitting in trial_data) and current ones (column only in
    trial_kinematics), this checks trial_data first and falls back to the
    companion kinematics file, auto-detected by swapping the
    'trial_data_' filename prefix for 'trial_kinematics_' unless
    kinematics_path is given explicitly. Never raises on a missing column
    or missing file -- move_samples just comes back None, and the caller
    (TrialFigure.build) already falls back to the trial's own raw sample
    count in that case.
    """
    KEYS = ["Block", "TrialNumInBlock", "Attempt"]

    def __init__(self, path, kinematics_path=None):
        self.frame = pd.read_csv(path) if path and Path(path).exists() else None
        if self.frame is not None:
            self.frame = normalize_trial_schema(self.frame)

        if kinematics_path is None and path:
            guess = Path(path).with_name(Path(path).name.replace("trial_data_", "trial_kinematics_", 1))
            kinematics_path = guess if guess.exists() else None
        self.kin_frame = pd.read_csv(kinematics_path) if kinematics_path and Path(kinematics_path).exists() else None

    @staticmethod
    def _match(frame, block, trial, attempt):
        row = frame[(frame.Block == block) & (frame.TrialNumInBlock == trial) & (frame.Attempt == attempt)]
        return row.iloc[0] if len(row) == 1 else None

    def lookup(self, block, trial, attempt):
        result = {"bar_size_deg": None, "move_samples": None,
                  "direction_correct": None, "is_correct": None, "error_type": None,
                  "decision_time_s": None, "execution_time_s": None, "total_time_s": None}

        if self.frame is not None:
            row = self._match(self.frame, block, trial, attempt)
            if row is not None:
                result["bar_size_deg"] = float(row["BarSizeVA_deg"])
                result["direction_correct"] = str(row["DirectionCorrect"])
                result["is_correct"] = int(row["IsCorrect"])
                result["error_type"] = int(row["ErrorType"])
                if "DecisionTime_s" in row.index:
                    result["decision_time_s"] = float(row["DecisionTime_s"])
                if "ExecutionTime_s" in row.index:
                    result["execution_time_s"] = float(row["ExecutionTime_s"])
                if "TotalTime_s" in row.index:
                    result["total_time_s"] = float(row["TotalTime_s"])
                if "NumMovementSamples" in row.index:
                    result["move_samples"] = int(row["NumMovementSamples"])

        if result["move_samples"] is None and self.kin_frame is not None:
            krow = self._match(self.kin_frame, block, trial, attempt)
            if krow is not None and "NumMovementSamples" in krow.index:
                result["move_samples"] = int(krow["NumMovementSamples"])

        return result


def estimate_screen_layout(dataset, info_table, target_dist=TARGET_DIST_PX,
                           target_radius=TARGET_RADIUS_PX, center_radius=CENTER_RADIUS_PX,
                           center_override=None):
    if center_override is not None:
        cx, cy = center_override
    else:
        directions = ["Right_0", "Up_90", "Left_180", "Down_270"]
        ends = {d: [] for d in directions}
        for trial in dataset.trials():
            info = info_table.lookup(trial.block, trial.trial, trial.attempt)
            d = info.get("direction_correct")
            if info.get("is_correct") == 1 and d in ends:
                ends[d].append([trial.x[-1], trial.y[-1]])
        means = {}
        for d, pts in ends.items():
            if pts:
                arr = np.asarray(pts)
                means[d] = (float(arr[:, 0].mean()), float(arr[:, 1].mean()))
        cx = cy = None
        if "Right_0" in means and "Left_180" in means:
            cx = (means["Right_0"][0] + means["Left_180"][0]) / 2.0
        if "Up_90" in means and "Down_270" in means:
            cy = (means["Up_90"][1] + means["Down_270"][1]) / 2.0
        if cx is None:
            xs = [p[0] for p in means.values()]
            cx = float(np.mean(xs)) if xs else None
        if cy is None:
            ys = [p[1] for p in means.values()]
            cy = float(np.mean(ys)) if ys else None
        if cx is None or cy is None:
            return None
    offsets = {"Right_0": (target_dist, 0.0), "Up_90": (0.0, -target_dist),
               "Left_180": (-target_dist, 0.0), "Down_270": (0.0, target_dist)}
    targets = {d: (cx + ox, cy + oy) for d, (ox, oy) in offsets.items()}
    return {"center": (cx, cy), "targets": targets,
            "target_radius": target_radius, "center_radius": center_radius}




## 3. RNN-ready dataset

The two EDA notebooks (section 1, per session; section 2, across sessions) describe each trial as a point (summary kinematics, psychometric
and chronometric fits). A recurrent model consumes the trial as a **sequence**, so
this section looks at the data the way the network will see it:

| Cell | What it adds | Which network needs it |
|---|---|---|
| 3.1 | Every session's full trajectory cut into one window per trial, on a uniform grid | both |
| 3.2 | Epoch durations and onsets (bar, cue, go, movement, target) | task network input schedule `u(t)` |
| 3.3 | Session inventory, schema check, split by session | training protocol |
| 3.4 | Condition-averaged velocity profiles and the hold-period noise floor | motor network targets `y(t)` |
| 3.5 | Time-resolved linear decoding, with a leakage control | baseline the recurrence must beat |
| 3.6 | Functional PCA of the velocity curves | size of the task-to-motor bottleneck |
| 3.7 | Tensor export (`rnn_dataset.npz`) | both |

The two-network design these cells prepare for: a continuous-time rate RNN that
receives the bar length and the target cue and outputs the category at every step
(many-to-many, loss masked to the post-go window), whose hidden state feeds through
a low-dimensional bottleneck into a GRU that outputs `vx(t), vy(t)` (many-to-many,
aligned).

Requires 0.1 (session inventory) and 0.2 (the signal-processing pipeline is
only used by the non-causal comparison in 3.5).

### 3.1. Trial windows on the RNN grid

Reads `trajectory_<runTag>.csv` (the full export, every epoch) for every session in the
inventory, joins it to `trial_data_<runTag>.csv` on `(Block, TrialNumInBlock, Attempt)`,
and resamples each trial to `RNN_DT_MS` over a window from `RNN_PRE_STIM_MS` before bar
onset to `RNN_POST_GO_MS` after target onset. Positions are centre-relative with y up,
velocities in cm/s.

`RNN_RESAMPLE_MODE` matters. The section 1.1 pipeline is a *smoother*: the Kalman RTS pass
and the zero-phase Butterworth both use future samples, which is fine for summary
kinematics but puts the coming movement into the samples before it. For a model whose
target is `y(t)`, that leaks the answer backwards in time (3.5 measures it). The default
here is PCHIP plus a one-directional Butterworth, which lags a little and never looks ahead.

In [ ]:
import glob as _glob
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.interpolate import PchipInterpolator

# ---------------------------------------------------------------------------
# 3.1  Trial timeline + resampled sequences, every session
#
# Everything in section 3 is built from the FULL trajectory export
# (trajectory_<runTag>.csv), not the movement-only cut, because the RNN needs
# the epochs the earlier sections ignore: the bar, the delays and the go.
# One pass over every session in the inventory produces
#   RNN_TRIALS : one row per trial with the epoch onsets and the trial labels
#   RNN_SEQ    : per-trial arrays on a uniform RNN_DT_MS grid
# and the later cells only read those two.
# ---------------------------------------------------------------------------
RNN_DT_MS = 10.0             # RNN time step; the task samples at ~9 ms
RNN_PRE_STIM_MS = 300.0      # window starts this long before bar onset
RNN_POST_GO_MS = 1500.0      # and ends this long after target onset (go)
RNN_MIN_TRIALS = 100         # sessions with fewer usable trials are excluded
RNN_MIN_ACCURACY = 0.5       # and so are sessions below this accuracy
RNN_EXCLUDE_SESSIONS = []    # run tags to drop by hand, e.g. ["sessROM_28-Aug-2026_15-06"]
# "causal" (default): PCHIP + one-directional Butterworth, no look-ahead.
# "plain": PCHIP only. "pipeline": section 1.1's non-causal smoother (see 3.5).
RNN_RESAMPLE_MODE = "causal"
RNN_CAUSAL_CUTOFF_HZ = 10.0

# TaskEpoch.m codes (centerTask/TaskEpoch.m). 14/15 are the optional delays.
RNN_EPOCH_NAMES = {1: "ENTER_CENTER", 2: "HOLD_START", 3: "HOLD", 4: "BAR", 5: "CUE",
                   6: "DECISION", 7: "MOVEMENT", 8: "TARGET_HOLD", 9: "REWARD",
                   10: "SUCCESS_FB", 11: "ERROR_FB", 12: "ITI", 13: "BOOKKEEP",
                   14: "STIM_DELAY", 15: "CUE_DELAY"}
STIM_EPOCH, GO_EPOCH, MOVE_EPOCH, HOLD_EPOCH_CODE, CENTER_HOLD_EPOCH = 4, 6, 7, 8, 3
# Trial-internal epochs whose durations define the RNN timeline, in task order.
RNN_TIMELINE_EPOCHS = [3, 4, 14, 5, 15, 6, 7, 8]

DIRECTION_ANGLE_DEG = {"Right_0": 0.0, "Up_90": 90.0, "Left_180": 180.0, "Down_270": 270.0}
DIRECTION_INDEX = {"Right_0": 0, "Up_90": 1, "Left_180": 2, "Down_270": 3}
RNN_CATEGORY_ID = {"ShortGroup": 0, "MidGroup": 1, "LongGroup": 2}
RNN_KEYS = ["Block", "TrialNumInBlock", "Attempt"]
_RNN_CM_PER_PX = globals().get("CM_PER_PX", 0.3108 / 10.0)


def _rnn_session_start(tag):
    fn = globals().get("session_timestamp")
    if callable(fn):
        return fn(tag)
    m = re.search(r"_(\d{2}-[A-Za-z]{3}-\d{4})_(\d{2})-(\d{2})$", str(tag))
    if not m:
        return pd.NaT
    return pd.to_datetime(f"{m.group(1)} {m.group(2)}:{m.group(3)}",
                          format="%d-%b-%Y %H:%M", errors="coerce")


def _rnn_inventory():
    """Session inventory: the one from 0.1 if it ran, else a bare scan of outputs/."""
    inv = globals().get("SESSION_INVENTORY")
    if isinstance(inv, pd.DataFrame) and not inv.empty:
        return inv
    fn = globals().get("discover_sessions")
    if callable(fn):
        inv = fn(verbose=False)
        if isinstance(inv, pd.DataFrame) and not inv.empty:
            return inv
    rows = []
    for cand in ["outputs", "/content/*/outputs", "../outputs"]:
        for root in sorted(_glob.glob(cand)):
            for d in sorted(p for p in Path(root).iterdir() if p.is_dir()):
                td = sorted(d.glob("trial_data_*.csv"))
                tj = sorted(d.glob("trajectory_sess*.csv")) or sorted(d.glob("trajectory_[!m]*.csv"))
                rows.append({"Session": d.name, "trial_data": str(td[0]) if td else None,
                             "trajectory": str(tj[0]) if tj else None})
            if rows:
                # same rule as 0.1 and 2.1: date order from the runTag, never
                # the file-system (alphabetical) order
                inv = pd.DataFrame(rows)
                inv["SessionStart"] = inv["Session"].map(_rnn_session_start)
                inv = inv.sort_values("SessionStart", kind="stable", na_position="last").reset_index(drop=True)
                inv.insert(0, "index", np.arange(1, len(inv) + 1))
                return inv
    return pd.DataFrame()


def _rnn_epoch_codes(series):
    if pd.api.types.is_numeric_dtype(series):
        return series.to_numpy(dtype=float)
    fn = globals().get("_epoch_to_code")
    if callable(fn):
        return series.map(fn).to_numpy(dtype=float)
    return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)


def _rnn_normalize_trials(td):
    fn = globals().get("normalize_trial_schema")
    return fn(td) if callable(fn) else td


def trial_epoch_onsets(traj):
    """One row per (Block, TrialNumInBlock, Attempt) with the first and last
    Time_ms of every epoch that trial visited, as on_<code> / off_<code>, plus
    n_rows and the Time_ms monotonicity check the trajectory schema promises."""
    rows = []
    for key, g in traj.groupby(RNN_KEYS, sort=False):
        t = g["Time_ms"].to_numpy(dtype=float)
        ep = g["_epoch"].to_numpy()
        rec = dict(zip(RNN_KEYS, (int(k) for k in key)))
        rec["n_rows"] = int(t.size)
        rec["t_first_ms"] = float(t[0])
        rec["t_last_ms"] = float(t[-1])
        rec["n_time_backsteps"] = int(np.sum(np.diff(t) < 0)) if t.size > 1 else 0
        for code in np.unique(ep[np.isfinite(ep)]):
            m = ep == code
            rec[f"on_{int(code)}"] = float(t[m].min())
            rec[f"off_{int(code)}"] = float(t[m].max())
        rows.append(rec)
    return pd.DataFrame(rows)


class _PlainResampler:
    """PCHIP onto the grid, nothing else. Causal in the sense that matters:
    a sample can only be shaped by its raw neighbours, never by the movement
    that follows 200 ms later."""
    name = "pchip-only"

    def __call__(self, t_ms, x, y, grid_ms):
        t, keep = np.unique(np.asarray(t_ms, float), return_index=True)
        x = np.asarray(x, float)[keep]
        y = np.asarray(y, float)[keep]
        if t.size < 2:
            return np.full((grid_ms.size, 2), np.nan)
        return np.column_stack([PchipInterpolator(t, x)(grid_ms), PchipInterpolator(t, y)(grid_ms)])


class _CausalResampler(_PlainResampler):
    """PCHIP, then a one-directional (lfilter) Butterworth low-pass: smooths
    the joystick noise with a small lag and NO look-ahead."""
    name = "pchip+causal-butter"

    def __init__(self, cutoff_hz=RNN_CAUSAL_CUTOFF_HZ, order=2):
        from scipy.signal import butter
        self.ba = butter(order, cutoff_hz / (0.5 * 1000.0 / RNN_DT_MS))

    def __call__(self, t_ms, x, y, grid_ms):
        from scipy.signal import lfilter, lfilter_zi
        xy = super().__call__(t_ms, x, y, grid_ms)
        if not np.isfinite(xy).all():
            return xy
        b, a = self.ba
        zi = lfilter_zi(b, a)
        return np.column_stack([lfilter(b, a, xy[:, k], zi=zi * xy[0, k])[0] for k in range(2)])


class _PipelineResampler:
    """Section 1.1's pipeline (Hampel -> Kalman+RTS -> PCHIP -> zero-phase
    Butterworth) run at the RNN step. NOT causal: the RTS smoother and the
    zero-phase filter both read the future, and 3.5 shows they leak the
    coming movement's direction into the pre-go samples. Kept for comparison
    with the earlier sections, not for building RNN targets."""
    name = "hampel-kalman-pchip-butter (non-causal)"

    def __init__(self):
        self.proc = TrajectoryProcessor(grid_dt_s=RNN_DT_MS / 1000.0)

    def __call__(self, t_ms, x, y, grid_ms):
        res = self.proc.process(t_ms, x, y)
        gt = res.grid_time_ms + float(np.min(t_ms))
        gxy = res.grid_xy
        if gt.size < 2:
            return np.full((grid_ms.size, 2), np.nan)
        return np.column_stack([np.interp(grid_ms, gt, gxy[:, 0]), np.interp(grid_ms, gt, gxy[:, 1])])


def _make_resampler(mode):
    if mode == "pipeline":
        if "TrajectoryProcessor" not in globals():
            print("RNN_RESAMPLE_MODE='pipeline' needs the 0.2 pipeline cell to have run; falling back to 'causal'.")
            return _CausalResampler()
        return _PipelineResampler()
    if mode == "plain":
        return _PlainResampler()
    return _CausalResampler()


RNN_RESAMPLER = _make_resampler(RNN_RESAMPLE_MODE)


def _rnn_sequence(g, t_stim, t_end, resampler):
    """Resample one trial's raw rows onto the RNN grid spanning
    [t_stim - PRE, t_end]. Returns grid (ms, absolute), xy (px, centre-relative,
    y up), v (cm/s), and a validity mask (grid points inside raw coverage)."""
    t = g["Time_ms"].to_numpy(dtype=float)
    x = g["X_px"].to_numpy(dtype=float)
    y = g["Y_px"].to_numpy(dtype=float)
    ep = g["_epoch"].to_numpy()
    t0 = t_stim - RNN_PRE_STIM_MS
    grid = np.arange(t0, t_end + 1e-9, RNN_DT_MS)
    pad = 5 * RNN_DT_MS
    m = (t >= t0 - pad) & (t <= t_end + pad)
    if m.sum() < 2:
        return grid, np.full((grid.size, 2), np.nan), np.full((grid.size, 2), np.nan), np.zeros(grid.size, bool)
    xy = resampler(t[m], x[m], y[m], grid)
    # centre = median cursor position while holding the centre, before the bar
    hm = m & (ep == CENTER_HOLD_EPOCH)
    if hm.sum() < 3:
        hm = m & (t < t_stim)
    cx = np.median(x[hm]) if hm.any() else np.median(x[m])
    cy = np.median(y[hm]) if hm.any() else np.median(y[m])
    xy_rel = np.column_stack([xy[:, 0] - cx, -(xy[:, 1] - cy)])  # screen y is down; flip
    v = np.gradient(xy_rel, RNN_DT_MS / 1000.0, axis=0) * _RNN_CM_PER_PX
    valid = (grid >= t[m].min()) & (grid <= t[m].max()) & np.isfinite(xy_rel).all(axis=1)
    return grid, xy_rel, v, valid


def build_rnn_trials(inventory, verbose=True):
    meta_rows, seqs = [], {}
    for _, r in inventory.iterrows():
        tag = r["Session"]
        if not r.get("trial_data") or not r.get("trajectory"):
            if verbose:
                print(f"  {tag}: missing trial_data or full trajectory; skipped.")
            continue
        td = _rnn_normalize_trials(pd.read_csv(r["trial_data"]))
        traj = pd.read_csv(r["trajectory"])
        traj["_epoch"] = _rnn_epoch_codes(traj["Epoch"])
        onsets = trial_epoch_onsets(traj)
        merged = td.merge(onsets, on=RNN_KEYS, how="left", validate="one_to_one")
        merged["Session"] = tag
        merged["SessionIndex"] = int(r["index"]) if "index" in r else np.nan
        merged["has_rz2idx"] = "RZ2Idx" in traj.columns
        merged["traj_rows"] = len(traj)
        groups = dict(tuple(traj.groupby(RNN_KEYS, sort=False)))
        n_seq = 0
        for i, row in merged.iterrows():
            t_stim = row.get(f"on_{STIM_EPOCH}", np.nan)
            t_go = row.get(f"on_{GO_EPOCH}", np.nan)
            reached_go = np.isfinite(t_stim) and np.isfinite(t_go)
            merged.at[i, "reached_go"] = bool(reached_go)
            if not reached_go:
                continue
            # the window closes RNN_POST_GO_MS after go, or when the trial's
            # rows run out, whichever is first
            t_end = min(t_go + RNN_POST_GO_MS, row["t_last_ms"])
            g = groups[(int(row["Block"]), int(row["TrialNumInBlock"]), int(row["Attempt"]))]
            grid, xy, v, valid = _rnn_sequence(g, t_stim, t_end, RNN_RESAMPLER)
            seqs[(tag, int(row["Block"]), int(row["TrialNumInBlock"]), int(row["Attempt"]))] = {
                "grid_ms": grid, "xy_px": xy, "v_cms": v, "valid": valid}
            merged.at[i, "n_steps"] = int(grid.size)
            merged.at[i, "n_valid"] = int(valid.sum())
            n_seq += 1
        meta_rows.append(merged)
        if verbose:
            print(f"  {tag}: {len(td)} trials, {n_seq} reached go, "
                  f"{len(traj)} trajectory rows, RZ2Idx={'yes' if 'RZ2Idx' in traj.columns else 'no'}")
    if not meta_rows:
        return pd.DataFrame(), {}
    meta = pd.concat(meta_rows, ignore_index=True, sort=False)
    meta["reached_go"] = meta["reached_go"].fillna(False).astype(bool)
    # times relative to bar onset (ms), which is t = 0 for every trial
    for code in RNN_TIMELINE_EPOCHS + [9, 10, 11]:
        on, off = f"on_{code}", f"off_{code}"
        if on in meta.columns:
            meta[f"{RNN_EPOCH_NAMES[code].lower()}_on_ms"] = meta[on] - meta[f"on_{STIM_EPOCH}"]
            meta[f"{RNN_EPOCH_NAMES[code].lower()}_dur_ms"] = meta[off] - meta[on]
    meta["go_ms"] = meta.get(f"on_{GO_EPOCH}") - meta[f"on_{STIM_EPOCH}"]
    meta["move_on_ms"] = meta.get(f"on_{MOVE_EPOCH}", np.nan) - meta[f"on_{STIM_EPOCH}"]
    meta["target_ms"] = meta.get(f"on_{HOLD_EPOCH_CODE}", np.nan) - meta[f"on_{STIM_EPOCH}"]
    meta["window_ms"] = meta["n_steps"] * RNN_DT_MS
    meta["category_id"] = meta["StimulusGroup"].map(RNN_CATEGORY_ID)
    meta["dir_correct_id"] = meta["DirectionCorrect"].map(DIRECTION_INDEX)
    meta["dir_chosen_id"] = meta["DirectionChosen"].map(DIRECTION_INDEX)
    return meta, seqs


_inv = _rnn_inventory()
if _inv.empty:
    print("No sessions found. Run 0.1 (or set OUTPUTS_DIR) and re-run this cell.")
    RNN_TRIALS, RNN_SEQ = pd.DataFrame(), {}
else:
    print(f"Resampler: {RNN_RESAMPLER.name}  (dt = {RNN_DT_MS:g} ms, "
          f"window = bar - {RNN_PRE_STIM_MS:g} ms .. go + {RNN_POST_GO_MS:g} ms)")
    RNN_TRIALS, RNN_SEQ = build_rnn_trials(_inv)
    if not RNN_TRIALS.empty:
        n_go = int(RNN_TRIALS["reached_go"].sum())
        print(f"\nRNN_TRIALS: {len(RNN_TRIALS)} trials in {RNN_TRIALS['Session'].nunique()} sessions; "
              f"{n_go} reached go and have a sequence in RNN_SEQ.")
        print(f"Window length: median {RNN_TRIALS['window_ms'].median():.0f} ms, "
              f"max {RNN_TRIALS['window_ms'].max():.0f} ms "
              f"-> T = {int(np.ceil(RNN_TRIALS['n_steps'].max()))} steps at {RNN_DT_MS:g} ms.")
        RNN_TRIALS.to_csv("rnn_trials.csv", index=False)

### 3.2. Epoch timeline

Durations of every trial-internal epoch and their onsets relative to bar onset. If
the bar-to-go interval is essentially fixed, the task network can be trained with one
schedule; if it varies, `u(t)` has to be built from each trial's own onsets (3.7 does
the latter regardless). The DECISION and MOVEMENT epochs are checked against the
`DecisionTime_s` and `ExecutionTime_s` the trial table logged.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 3.2  Epoch timeline: how long each epoch lasts, trial by trial
#
# The task network's input u(t) is a schedule (bar on, delay, cue, go), so
# the first thing the RNN needs is the distribution of those durations. This
# also decides between real per-trial epoch times and one fixed schedule.
# ---------------------------------------------------------------------------
_tl = RNN_TRIALS[RNN_TRIALS["reached_go"]].copy()
_dur_cols = [f"{RNN_EPOCH_NAMES[c].lower()}_dur_ms" for c in RNN_TIMELINE_EPOCHS
             if f"{RNN_EPOCH_NAMES[c].lower()}_dur_ms" in _tl.columns]
_on_cols = [f"{RNN_EPOCH_NAMES[c].lower()}_on_ms" for c in RNN_TIMELINE_EPOCHS
            if f"{RNN_EPOCH_NAMES[c].lower()}_on_ms" in _tl.columns]

print("Epoch durations (ms), all sessions pooled, trials that reached go:")
_desc = _tl[_dur_cols].describe(percentiles=[0.05, 0.5, 0.95]).T
_desc["n_missing"] = _tl[_dur_cols].isna().sum()
print(_desc[["count", "mean", "std", "5%", "50%", "95%", "max", "n_missing"]].round(1).to_string())

print("\nEpoch onsets relative to bar onset (ms), median per session:")
print(_tl.groupby("Session")[_on_cols + ["go_ms", "move_on_ms", "target_ms"]].median().round(0).to_string())

# Consistency: the DECISION epoch on the trajectory side should be the
# DecisionTime_s the trial table logged, and MOVEMENT the ExecutionTime_s.
if "decision_dur_ms" in _tl.columns and "DecisionTime_s" in _tl.columns:
    d = _tl["decision_dur_ms"] - 1000.0 * _tl["DecisionTime_s"]
    print(f"\nDECISION epoch vs DecisionTime_s: median gap {d.median():.1f} ms, "
          f"95% of trials within +-{d.abs().quantile(0.95):.1f} ms "
          "(the epoch is sampled at the cursor rate, so a gap of one sample is expected).")
if "movement_dur_ms" in _tl.columns and "ExecutionTime_s" in _tl.columns:
    d = _tl["movement_dur_ms"] - 1000.0 * _tl["ExecutionTime_s"]
    print(f"MOVEMENT epoch vs ExecutionTime_s: median gap {d.median():.1f} ms, "
          f"95% within +-{d.abs().quantile(0.95):.1f} ms.")

# Is the schedule fixed? The bar-to-go interval is what the task network has
# to bridge in memory; if it barely varies, one fixed schedule will do.
_go = _tl["go_ms"]
print(f"\nBar onset -> go: median {_go.median():.0f} ms, IQR {_go.quantile(0.25):.0f}-{_go.quantile(0.75):.0f} ms, "
      f"range {_go.min():.0f}-{_go.max():.0f} ms.")
print(f"Go -> movement onset (RT): median {_tl['move_on_ms'].sub(_go).median():.0f} ms; "
      f"go -> target: median {_tl['target_ms'].sub(_go).median():.0f} ms.")
_T = int(RNN_TRIALS["n_steps"].max())
_T95 = int(np.ceil(RNN_TRIALS["n_steps"].quantile(0.95)))
print(f"Sequence length: T_max = {_T} steps, T_95 = {_T95} steps at {RNN_DT_MS:g} ms "
      f"(pad to T_max with a validity mask, or truncate at T_95).")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ax = axes[0]
_lab = [c.replace("_dur_ms", "") for c in _dur_cols]
ax.boxplot([_tl[c].dropna().to_numpy() for c in _dur_cols], showfliers=False)
ax.set_xticks(np.arange(1, len(_lab) + 1))
ax.set_xticklabels(_lab)
ax.set_ylabel("duration (ms)")
ax.set_title("Epoch durations (all sessions)")
ax.tick_params(axis="x", rotation=45)

ax = axes[1]
_order = _tl.groupby("Session")["SessionIndex"].first().sort_values().index
_bars = _tl.groupby("Session")[_on_cols].median().loc[_order]
_left = np.zeros(len(_bars))
for c in _on_cols + ["go_ms"]:
    if c not in _bars.columns and c == "go_ms":
        _bars[c] = _tl.groupby("Session")["go_ms"].median().loc[_order]
for j, c in enumerate(_on_cols):
    nxt = _on_cols[j + 1] if j + 1 < len(_on_cols) else None
    end = _bars[nxt] if nxt is not None else _tl.groupby("Session")["target_ms"].median().loc[_order]
    width = (end - _bars[c]).clip(lower=0)
    ax.barh(np.arange(len(_bars)), width, left=_bars[c], label=c.replace("_on_ms", ""))
ax.set_yticks(np.arange(len(_bars)))
ax.set_yticklabels([s.replace("sessROM_", "") for s in _bars.index], fontsize=7)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("ms from bar onset")
ax.set_title("Median trial timeline per session")
ax.legend(fontsize=7, ncol=2)

ax = axes[2]
ax.hist(_tl["window_ms"].dropna(), bins=30, color="#888888")
ax.axvline(_T95 * RNN_DT_MS, color="r", ls="--", label=f"95th pct = {_T95} steps")
ax.set_xlabel("RNN window length (ms)")
ax.set_ylabel("trials")
ax.set_title("Sequence length distribution")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 3.3. Session inventory and split

Which sessions enter the dataset and why, plus a chronological split **by session**.
A trial-level split would let session-specific bias (fatigue, joystick calibration,
the two-clock timing bug) leak from train into test. The per-condition counts show
whether every length x direction cell has enough correct trials for the
condition-mean targets used in 3.4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 3.3  Session inventory, schema check, and the train / val / test split
#
# Sessions differ in schema generation and in quality. This cell decides,
# per session, whether it goes into the dataset, and assigns the split BY
# SESSION (chronological), which is the only honest split at this volume:
# a trial-level split leaks session-specific bias into the test set.
# ---------------------------------------------------------------------------
RNN_TEST_SESSIONS = 1     # most recent session(s) held out for test
RNN_VAL_SESSIONS = 1      # the one(s) before that for validation

_rows = []
for tag, g in RNN_TRIALS.groupby("Session", sort=False):
    ok = g[g["reached_go"]]
    per_cond = ok[ok["IsCorrect"] == 1].groupby(["BarSizeVA_deg", "DirectionCorrect"]).size()
    _rows.append({
        "Session": tag,
        "SessionIndex": g["SessionIndex"].iloc[0],
        "n_trials": len(g),
        "n_reached_go": len(ok),
        "accuracy": float(ok["IsCorrect"].mean()) if len(ok) else np.nan,
        "n_lengths": int(g["BarSizeVA_deg"].nunique()),
        "session_mode": g["SessionMode"].iloc[0] if "SessionMode" in g.columns and g["SessionMode"].notna().any() else "(absent)",
        "n_attempt_gt1": int((g["Attempt"] > 1).sum()),
        "n_early_exit": int((pd.to_numeric(g.get("ChosenTarget", 0), errors="coerce") <= 0).sum()),
        "has_rz2idx": bool(g["has_rz2idx"].iloc[0]),
        "time_backsteps": int(g["n_time_backsteps"].fillna(0).sum()),
        "traj_rows": int(g["traj_rows"].iloc[0]),
        "min_per_cond": int(per_cond.min()) if len(per_cond) else 0,
        "n_cond": int(len(per_cond)),
    })
RNN_SESSIONS = pd.DataFrame(_rows).sort_values("SessionIndex").reset_index(drop=True)

# fs from the raw rows: median step between samples, per session
_fs = {}
for tag, g in RNN_TRIALS.groupby("Session"):
    _fs[tag] = float(g["n_rows"].sum() / ((g["t_last_ms"] - g["t_first_ms"]).sum() / 1000.0))
RNN_SESSIONS["fs_hz"] = RNN_SESSIONS["Session"].map(_fs).round(1)

# inclusion
RNN_SESSIONS["excluded_because"] = ""
_m = RNN_SESSIONS["n_reached_go"] < RNN_MIN_TRIALS
RNN_SESSIONS.loc[_m, "excluded_because"] += f"<{RNN_MIN_TRIALS} trials; "
_m = RNN_SESSIONS["accuracy"] < RNN_MIN_ACCURACY
RNN_SESSIONS.loc[_m, "excluded_because"] += f"accuracy<{RNN_MIN_ACCURACY}; "
_m = RNN_SESSIONS["Session"].isin(RNN_EXCLUDE_SESSIONS)
RNN_SESSIONS.loc[_m, "excluded_because"] += "listed in RNN_EXCLUDE_SESSIONS; "
RNN_SESSIONS["included"] = RNN_SESSIONS["excluded_because"] == ""

# split: chronological, by session
_inc = RNN_SESSIONS[RNN_SESSIONS["included"]]["Session"].tolist()
_split = {s: "train" for s in _inc}
for s in _inc[-RNN_TEST_SESSIONS:]:
    _split[s] = "test"
for s in _inc[-(RNN_TEST_SESSIONS + RNN_VAL_SESSIONS):-RNN_TEST_SESSIONS]:
    _split[s] = "val"
RNN_SESSIONS["split"] = RNN_SESSIONS["Session"].map(_split).fillna("excluded")
RNN_TRIALS["split"] = RNN_TRIALS["Session"].map(_split).fillna("excluded")
RNN_TRIALS["included"] = RNN_TRIALS["split"].ne("excluded") & RNN_TRIALS["reached_go"]

_show = ["SessionIndex", "Session", "n_trials", "accuracy", "n_lengths", "session_mode",
         "fs_hz", "has_rz2idx", "time_backsteps", "min_per_cond", "n_cond", "split", "excluded_because"]
print(RNN_SESSIONS[_show].round(3).to_string(index=False))
print(f"\nIncluded: {RNN_TRIALS['included'].sum()} trials from {len(_inc)} sessions "
      f"(train/val/test = "
      + "/".join(str(int((RNN_TRIALS.loc[RNN_TRIALS['included'], 'split'] == s).sum())) for s in ("train", "val", "test"))
      + ").")
if (RNN_SESSIONS["time_backsteps"] > 0).any():
    print("WARNING: some sessions have Time_ms going backwards inside a trial; "
          "those rows were sorted by the resampler, but check RZ2_JOYSTICK.md for the two-clock issue.")
if not RNN_SESSIONS["has_rz2idx"].any():
    print("Note: no exported trajectory carries RZ2Idx yet (added 2026-09-04), so Time_ms "
          "is the only time base and the sample rate cannot be re-derived offline.")

# Trials per (length x direction) condition across the included sessions.
# The condition-averaged velocity profiles in 3.4 are the first training
# target for the motor network, so every cell here needs a healthy count.
_ok = RNN_TRIALS[RNN_TRIALS["included"] & (RNN_TRIALS["IsCorrect"] == 1)]
RNN_COND_COUNTS = (_ok.groupby(["BarSizeVA_deg", "DirectionCorrect"]).size()
                   .unstack("DirectionCorrect").reindex(columns=list(DIRECTION_INDEX)).fillna(0).astype(int))
print("\nCorrect trials per length x direction, included sessions pooled:")
print(RNN_COND_COUNTS.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
im = ax.imshow(RNN_COND_COUNTS.to_numpy(), aspect="auto", cmap="viridis")
ax.set_xticks(range(RNN_COND_COUNTS.shape[1]))
ax.set_xticklabels(RNN_COND_COUNTS.columns, fontsize=8)
ax.set_yticks(range(RNN_COND_COUNTS.shape[0]))
ax.set_yticklabels([f"{v:.2f}" for v in RNN_COND_COUNTS.index], fontsize=8)
ax.set_ylabel("bar length (deg VA)")
ax.set_title("Correct trials per condition (pooled)")
for (i, j), v in np.ndenumerate(RNN_COND_COUNTS.to_numpy()):
    ax.text(j, i, str(v), ha="center", va="center", fontsize=7,
            color="w" if v < RNN_COND_COUNTS.to_numpy().max() / 2 else "k")
plt.colorbar(im, ax=ax)

ax = axes[1]
_per = (RNN_TRIALS[RNN_TRIALS["reached_go"]].groupby(["SessionIndex", "split"]).size()
        .unstack("split").reindex(columns=["train", "val", "test", "excluded"]).fillna(0))
_per.plot.bar(stacked=True, ax=ax, color={"train": "#4C72B0", "val": "#DD8452", "test": "#55A868", "excluded": "#BBBBBB"})
ax.set_xlabel("session index")
ax.set_ylabel("trials")
ax.set_title("Split by session (chronological)")
plt.tight_layout()
plt.show()

### 3.4. Velocity profiles per condition and the hold noise floor

`vx(t), vy(t)` per (length x direction), aligned to go and to movement onset,
projected onto the target axis. These condition means are the motor network's
first training target, before it is asked to fit single trials. The speed during the
centre hold is the "zero" the network must produce before go, and the level below
which a lower loss means nothing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 3.4  Condition-averaged velocity profiles and the hold-period noise floor
#
# vx(t), vy(t) per (length x direction), aligned to go and to movement onset.
# These are the motor network's first training target (mean profile per
# condition), and the pre-go speed is its "zero" target and the error floor.
# ---------------------------------------------------------------------------
PROFILE_PRE_MS = 200.0     # window shown before the alignment event
PROFILE_POST_MS = 800.0    # and after it
PROFILE_ONLY_CORRECT = True

_n_pre = int(PROFILE_PRE_MS / RNN_DT_MS)
_n_post = int(PROFILE_POST_MS / RNN_DT_MS)
_L = _n_pre + _n_post
PROFILE_T_MS = (np.arange(_L) - _n_pre) * RNN_DT_MS


def aligned_velocity(row, align="go"):
    """Velocity (L x 2, cm/s) of one trial cut around go or movement onset;
    NaN where the trial has no sample."""
    seq = RNN_SEQ.get((row["Session"], int(row["Block"]), int(row["TrialNumInBlock"]), int(row["Attempt"])))
    if seq is None:
        return None
    t_rel = seq["grid_ms"] - seq["grid_ms"][0]                    # ms from window start
    ev = RNN_PRE_STIM_MS + (row["go_ms"] if align == "go" else row["move_on_ms"])
    if not np.isfinite(ev):
        return None
    k = int(round(ev / RNN_DT_MS))
    out = np.full((_L, 2), np.nan)
    lo, hi = k - _n_pre, k + _n_post
    src_lo, src_hi = max(lo, 0), min(hi, seq["v_cms"].shape[0])
    if src_hi <= src_lo:
        return None
    v = seq["v_cms"].copy()
    v[~seq["valid"]] = np.nan
    out[src_lo - lo: src_hi - lo] = v[src_lo:src_hi]
    return out


def stack_aligned(df, align="go"):
    mats, keep = [], []
    for i, row in df.iterrows():
        m = aligned_velocity(row, align)
        if m is not None:
            mats.append(m)
            keep.append(i)
    return (np.stack(mats) if mats else np.empty((0, _L, 2))), df.loc[keep]


_sel = RNN_TRIALS[RNN_TRIALS["included"]]
if PROFILE_ONLY_CORRECT:
    _sel = _sel[_sel["IsCorrect"] == 1]
RNN_V_GO, _meta_go = stack_aligned(_sel, "go")
RNN_V_MOVE, _meta_move = stack_aligned(_sel, "move")
print(f"Aligned velocity stacks: go {RNN_V_GO.shape}, movement onset {RNN_V_MOVE.shape} "
      f"(trials x {_L} steps x [vx, vy]).")

# --- hold-period noise floor -------------------------------------------------
_speed_go = np.hypot(RNN_V_GO[..., 0], RNN_V_GO[..., 1])
_pre = _speed_go[:, :_n_pre]                     # 200 ms before go
_hold = []
for _, row in _sel.iterrows():
    seq = RNN_SEQ.get((row["Session"], int(row["Block"]), int(row["TrialNumInBlock"]), int(row["Attempt"])))
    if seq is None:
        continue
    k = int(RNN_PRE_STIM_MS / RNN_DT_MS)
    v = seq["v_cms"][:k][seq["valid"][:k]]
    _hold.append(np.hypot(v[:, 0], v[:, 1]))
_hold = np.concatenate(_hold) if _hold else np.array([])
RNN_HOLD_NOISE = {
    "pre_bar_speed_cms_p50": float(np.nanmedian(_hold)) if _hold.size else np.nan,
    "pre_bar_speed_cms_p95": float(np.nanpercentile(_hold, 95)) if _hold.size else np.nan,
    "pre_go_speed_cms_p50": float(np.nanmedian(_pre)),
    "pre_go_speed_cms_p95": float(np.nanpercentile(_pre, 95)),
}
print("\nHold-period speed (cm/s), the motor network's 'zero' target:")
for k, v in RNN_HOLD_NOISE.items():
    print(f"  {k:26s} {v:.3f}")
print("  A trained network whose pre-go MSE is below the square of the p95 value is at the data's floor.")

# --- condition means ---------------------------------------------------------
_lengths = sorted(_sel["BarSizeVA_deg"].unique())
_cat_of = _sel.groupby("BarSizeVA_deg")["StimulusGroup"].agg(lambda s: s.mode().iloc[0])
RNN_PROFILES = {}   # (align, length, direction) -> dict(mean, sd, n)
for align, V, meta in (("go", RNN_V_GO, _meta_go), ("move", RNN_V_MOVE, _meta_move)):
    for (ln, dr), idx in meta.groupby(["BarSizeVA_deg", "DirectionCorrect"]).indices.items():
        sub = V[idx]
        RNN_PROFILES[(align, ln, dr)] = {"mean": np.nanmean(sub, 0), "sd": np.nanstd(sub, 0), "n": len(idx)}

_cats = list(RNN_CATEGORY_ID)
_dirs = list(DIRECTION_INDEX)
_cmap = plt.get_cmap("viridis")
for align, title in (("go", "aligned to go (target onset)"), ("move", "aligned to movement onset")):
    fig, axes = plt.subplots(len(_cats), len(_dirs), figsize=(16, 9), sharex=True, sharey=True)
    for i, cat in enumerate(_cats):
        lens = [ln for ln in _lengths if _cat_of.get(ln) == cat]
        for j, dr in enumerate(_dirs):
            ax = axes[i, j]
            for k, ln in enumerate(lens):
                p = RNN_PROFILES.get((align, ln, dr))
                if p is None:
                    continue
                col = _cmap(0.15 + 0.7 * k / max(len(lens) - 1, 1))
                ang = np.deg2rad(DIRECTION_ANGLE_DEG[dr])
                # speed along the target axis: positive = toward the target
                along = p["mean"][:, 0] * np.cos(ang) + p["mean"][:, 1] * np.sin(ang)
                ax.plot(PROFILE_T_MS, along, color=col, lw=1.4, label=f"{ln:.2f} (n={p['n']})")
            ax.axvline(0, color="k", lw=0.7, ls="--")
            ax.axhline(0, color="k", lw=0.5)
            if i == 0:
                ax.set_title(dr)
            if j == 0:
                ax.set_ylabel(f"{cat}\nspeed toward target (cm/s)")
            if i == len(_cats) - 1:
                ax.set_xlabel("ms")
            ax.legend(fontsize=6, loc="upper left")
    fig.suptitle(f"Mean velocity along the target axis per length x direction, {title} "
                 f"({'correct trials' if PROFILE_ONLY_CORRECT else 'all trials'})")
    plt.tight_layout()
    plt.show()

# variability: per-condition SD envelope as a fraction of the peak, which is
# how much of the trajectory a condition-mean target can ever explain
_rows = []
for (align, ln, dr), p in RNN_PROFILES.items():
    if align != "move":
        continue
    spd = np.hypot(p["mean"][:, 0], p["mean"][:, 1])
    _rows.append({"length": ln, "direction": dr, "n": p["n"], "peak_cms": float(np.nanmax(spd)),
                  "sd_at_peak_cms": float(np.nanmean(p["sd"][np.nanargmax(spd)]))})
RNN_PROFILE_SUMMARY = pd.DataFrame(_rows)
RNN_PROFILE_SUMMARY["cv_at_peak"] = RNN_PROFILE_SUMMARY["sd_at_peak_cms"] / RNN_PROFILE_SUMMARY["peak_cms"]
print("\nPer-condition peak speed and its across-trial SD (movement-aligned):")
print(RNN_PROFILE_SUMMARY.round(3).sort_values(["direction", "length"]).to_string(index=False))

### 3.5. Time-resolved decoding

A linear read-out of the trajectory up to a growing horizon after go and after
movement onset. This is the many-to-many baseline: a GRU on the same windows has to
beat these curves, or the sequential component is not contributing. The leakage
control decodes the chosen direction from the 200 ms *before* go, where nothing on
screen distinguishes the directions yet: it must be at chance, and it is only at
chance with the causal resampler.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

# ---------------------------------------------------------------------------
# 3.5  Time-resolved decoding: how much does the trajectory say, and when?
#
# Section 1.10 decodes from whole-trial summary features. Here the decoder
# only sees the trajectory up to a growing horizon after go (and after
# movement onset), so the curve is the many-to-many baseline: at each step,
# what a linear read-out of the movement so far can tell about the trial.
# A recurrent model has to beat this curve, or the recurrence is not earning
# its keep.
# ---------------------------------------------------------------------------
DECODE_HORIZONS_MS = np.arange(50, 651, 50)
DECODE_TARGETS_T = ["DirectionChosen", "StimulusGroup", "IsCorrect"]
DECODE_FOLDS = 5
DECODE_SHUFFLE_CONTROL = True
DECODE_SEED_T = 0

_dec = RNN_TRIALS[RNN_TRIALS["included"]]
_V, _meta = stack_aligned(_dec, "go")
_Vm, _meta_m = stack_aligned(_dec, "move")


def _features_upto(V, n_steps):
    """Flatten vx, vy from the alignment event to n_steps after it, plus the
    cumulative displacement at the horizon; NaN -> 0 (trial ended)."""
    seg = V[:, _n_pre:_n_pre + n_steps, :]
    disp = np.nancumsum(np.nan_to_num(seg), axis=1)[:, -1, :] * (RNN_DT_MS / 1000.0)
    X = np.concatenate([np.nan_to_num(seg).reshape(len(V), -1), disp], axis=1)
    return X


def decode_curve(V, meta, target, horizons=DECODE_HORIZONS_MS, shuffle=False, seed=DECODE_SEED_T):
    y = meta[target].to_numpy()
    ok = pd.notna(y)
    V, y = V[ok], y[ok]
    if shuffle:
        y = np.random.default_rng(seed).permutation(y)
    classes, counts = np.unique(y, return_counts=True)
    n_splits = max(2, min(DECODE_FOLDS, int(counts.min())))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for h in horizons:
        X = _features_upto(V, int(h / RNN_DT_MS))
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5))
        sc = cross_val_score(clf, X, y, cv=cv, scoring="balanced_accuracy")
        out.append({"horizon_ms": float(h), "balanced_acc": float(sc.mean()), "sd": float(sc.std()),
                    "n": int(len(y)), "n_classes": int(len(classes)), "shuffled": shuffle})
    return pd.DataFrame(out)


RNN_DECODE_CURVES = {}
fig, axes = plt.subplots(1, len(DECODE_TARGETS_T), figsize=(5 * len(DECODE_TARGETS_T), 4), sharey=True)
for ax, target in zip(np.atleast_1d(axes), DECODE_TARGETS_T):
    for align, V, meta, col in (("go", _V, _meta, "#4C72B0"), ("move", _Vm, _meta_m, "#DD8452")):
        c = decode_curve(V, meta, target)
        RNN_DECODE_CURVES[(target, align)] = c
        ax.errorbar(c["horizon_ms"], c["balanced_acc"], yerr=c["sd"], color=col, marker="o", ms=3,
                    capsize=2, label=f"from {align}")
        if DECODE_SHUFFLE_CONTROL and align == "go":
            s = decode_curve(V, meta, target, horizons=DECODE_HORIZONS_MS[::3], shuffle=True)
            ax.plot(s["horizon_ms"], s["balanced_acc"], color="#999999", ls=":", marker="x", label="shuffled labels")
    ax.axhline(1.0 / c["n_classes"].iloc[0], color="k", lw=0.7, ls="--", label="chance")
    ax.set_title(f"{target}  (n={c['n'].iloc[0]}, {c['n_classes'].iloc[0]} classes)")
    ax.set_xlabel("horizon after alignment event (ms)")
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7)
np.atleast_1d(axes)[0].set_ylabel("balanced accuracy (CV)")
fig.suptitle("Linear decoding from the trajectory up to a horizon")
plt.tight_layout()
plt.show()

# --- leakage control ---------------------------------------------------------
# Decoding the chosen direction from the 200 ms BEFORE go must sit at chance:
# the targets are not on screen yet, so nothing in the cursor can know the
# direction. If it is above chance, the resampling looked into the future.
# Section 1.1's smoother (Kalman RTS + zero-phase Butterworth) does exactly
# that, which is why RNN_RESAMPLE_MODE defaults to "causal" in 3.1.
def _pre_go_control(V, meta, target="DirectionChosen", ms=200, seed=DECODE_SEED_T):
    y = meta[target].to_numpy()
    ok = pd.notna(y)
    X = np.nan_to_num(V[ok, _n_pre - int(ms / RNN_DT_MS):_n_pre, :]).reshape(ok.sum(), -1)
    cv = StratifiedKFold(n_splits=DECODE_FOLDS, shuffle=True, random_state=seed)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5))
    return float(cross_val_score(clf, X, y[ok], cv=cv, scoring="balanced_accuracy").mean())

RNN_LEAK_CONTROL = {RNN_RESAMPLER.name: _pre_go_control(_V, _meta)}
if "TrajectoryProcessor" in globals() and not isinstance(RNN_RESAMPLER, _PipelineResampler):
    # re-resample ONE session with the non-causal pipeline for the comparison
    _one = _rnn_inventory()
    _one = _one[_one["Session"] == _meta["Session"].iloc[-1]]
    _saved = (RNN_TRIALS, RNN_SEQ, RNN_RESAMPLER)
    RNN_RESAMPLER = _PipelineResampler()
    _T1, _S1 = build_rnn_trials(_one, verbose=False)
    _T1["included"] = _T1["reached_go"]
    RNN_TRIALS, RNN_SEQ = _T1, _S1
    _V1, _m1 = stack_aligned(_T1[_T1["included"]], "go")
    RNN_LEAK_CONTROL[RNN_RESAMPLER.name + f" [{_one['Session'].iloc[0]}]"] = _pre_go_control(_V1, _m1)
    RNN_TRIALS, RNN_SEQ, RNN_RESAMPLER = _saved
print("Leakage control: DirectionChosen decoded from the 200 ms BEFORE go (chance = 0.25):")
for k, v in RNN_LEAK_CONTROL.items():
    flag = "  <- reads the future" if v > 0.35 else ""
    print(f"  {k:55s} {v:.3f}{flag}")

print("\nBalanced accuracy by horizon (from go):")
_tab = pd.DataFrame({t: RNN_DECODE_CURVES[(t, "go")].set_index("horizon_ms")["balanced_acc"] for t in DECODE_TARGETS_T})
print(_tab.round(3).to_string())
print("\nReading: DirectionChosen should saturate quickly once the cursor leaves the centre. "
      "StimulusGroup is only decodable through movement vigour (the category-to-position map is "
      "shuffled per trial), so its curve is the honest ceiling for a category read-out from "
      "kinematics alone. If a GRU trained on the same windows does not beat these curves, the "
      "sequential component adds nothing over a linear read-out of the same samples.")

### 3.6. Functional PCA of the velocity curves

PCA over whole `(vx, vy)` curves in the movement window, so the components are
temporal shapes rather than scalar features. The number of components for 90-95 %
of the variance is the dimensionality the motor output actually has, and therefore
the natural size for the bottleneck between the task network and the motor network.
Rotating into the target frame removes the four directions, which the cue already
provides.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ---------------------------------------------------------------------------
# 3.6  Functional PCA of the velocity profiles: how many dimensions does the
#      motor output really have?
#
# Section 1.3 ran PCA on summary features. Here each trial is one curve
# (vx, vy over the movement-aligned window), so the components are temporal
# shapes. The number of components for 90-95 % variance is the size the
# bottleneck between the task network and the motor network should have,
# and it is the number to compare with the neural population later.
# ---------------------------------------------------------------------------
FPCA_WINDOW_MS = (0.0, 600.0)     # movement-aligned window fed to the PCA
FPCA_ROTATE_TO_TARGET = True      # also run it with velocities in the target frame

_a = int((FPCA_WINDOW_MS[0] + PROFILE_PRE_MS) / RNN_DT_MS)
_b = int((FPCA_WINDOW_MS[1] + PROFILE_PRE_MS) / RNN_DT_MS)
_Vf = RNN_V_MOVE[:, _a:_b, :]
_full = np.isfinite(_Vf).all(axis=(1, 2))
_Vf, _mf = _Vf[_full], _meta_move[_full]
print(f"Trials with a complete {FPCA_WINDOW_MS[0]:.0f}-{FPCA_WINDOW_MS[1]:.0f} ms movement window: "
      f"{_full.sum()} of {len(_full)}.")


def _rotate_to_target(V, meta):
    ang = np.deg2rad(meta["DirectionCorrect"].map(DIRECTION_ANGLE_DEG).to_numpy())
    c, s = np.cos(ang)[:, None], np.sin(ang)[:, None]
    along = V[..., 0] * c + V[..., 1] * s
    across = -V[..., 0] * s + V[..., 1] * c
    return np.stack([along, across], axis=-1)


def functional_pca(V, n_max=20):
    X = V.reshape(len(V), -1)
    p = PCA(n_components=min(n_max, X.shape[0], X.shape[1])).fit(X)
    cum = np.cumsum(p.explained_variance_ratio_)
    return p, cum


RNN_FPCA = {}
_variants = [("screen frame", _Vf)]
if FPCA_ROTATE_TO_TARGET:
    _variants.append(("target frame", _rotate_to_target(_Vf, _mf)))

fig, axes = plt.subplots(1, 1 + len(_variants), figsize=(5 * (1 + len(_variants)), 4))
for name, V in _variants:
    p, cum = functional_pca(V)
    n90 = int(np.searchsorted(cum, 0.90) + 1)
    n95 = int(np.searchsorted(cum, 0.95) + 1)
    RNN_FPCA[name] = {"pca": p, "cum": cum, "n90": n90, "n95": n95}
    axes[0].plot(np.arange(1, len(cum) + 1), cum, marker="o", ms=3, label=f"{name}: 90%={n90}, 95%={n95}")
axes[0].axhline(0.9, color="k", ls=":", lw=0.7)
axes[0].axhline(0.95, color="k", ls=":", lw=0.7)
axes[0].set_xlabel("components")
axes[0].set_ylabel("cumulative variance explained")
axes[0].set_title("Functional PCA of (vx, vy) curves")
axes[0].legend(fontsize=8)

_t = np.arange(_b - _a) * RNN_DT_MS + FPCA_WINDOW_MS[0]
for ax, (name, V) in zip(axes[1:], _variants):
    p = RNN_FPCA[name]["pca"]
    comps = p.components_[:3].reshape(3, _b - _a, 2)
    lab = ("along", "across") if name == "target frame" else ("vx", "vy")
    for k in range(3):
        ax.plot(_t, comps[k, :, 0], color=f"C{k}", label=f"PC{k + 1} {lab[0]} ({p.explained_variance_ratio_[k]:.0%})")
        ax.plot(_t, comps[k, :, 1], color=f"C{k}", ls="--", label=f"PC{k + 1} {lab[1]}")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xlabel("ms from movement onset")
    ax.set_title(f"First components, {name}")
    ax.legend(fontsize=6, ncol=2)
plt.tight_layout()
plt.show()

for name, r in RNN_FPCA.items():
    print(f"{name:13s}: {r['n90']} components for 90 %, {r['n95']} for 95 % of the variance.")
_n_bott = RNN_FPCA.get("target frame", RNN_FPCA["screen frame"])["n90"]
print(f"\nSuggested bottleneck between the task network and the motor network: ~{_n_bott} units "
      f"(90 % of the movement-shape variance in the target frame). In the screen frame the count "
      f"includes the 4 directions themselves, which the cue already gives the motor network.")

# component scores vs the stimulus: does any motor dimension carry the length
# (vigour), which is what a category read-out from kinematics relies on?
_p = RNN_FPCA[_variants[-1][0]]["pca"]
_scores = _p.transform(_variants[-1][1].reshape(len(_Vf), -1))[:, :5]
_corr = [np.corrcoef(_scores[:, k], _mf["BarSizeVA_deg"].to_numpy(float))[0, 1] for k in range(_scores.shape[1])]
print("Correlation of the first 5 component scores with bar length:", np.round(_corr, 3))

### 3.7. Tensor export

Writes `rnn_dataset.npz` with `u (B, T, 6)`, `y (B, T, 2)`, `xy`, `valid`, `mask_go`,
`epoch_on` and the per-trial labels, plus `rnn_trials_included.csv`. Padded to the
longest included trial with a validity mask; every array shares the same time axis,
with bar onset at step `RNN_PRE_STIM_MS / RNN_DT_MS`.

One limitation comes from the task, not the notebook: `trial_data` records only the
correct target's direction (and `foil_events` the foil's, on error trials). The full
three-target layout of a trial is not exported, so the cue input is the correct
direction. A model that must *find* the correct target among three coloured ones
needs a layout export added to `CenterOutTask.m`.

In [ ]:
import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 3.7  Export the tensor-ready dataset (rnn_dataset.npz + rnn_trials.csv)
#
# Layout, B trials x T steps:
#   u        (B, T, 6)  task-network input: [length, dir one-hot x4, go]
#                       length is the normalised bar length, on during BAR;
#                       the direction one-hot and go switch on at target onset
#   y        (B, T, 2)  motor target: vx, vy in cm/s (centre-relative, y up)
#   xy       (B, T, 2)  cursor position in px, centre-relative, y up
#   valid    (B, T)     1 where the trial has a cursor sample (padding = 0)
#   mask_go  (B, T)     1 from go onward -- the task network's output mask
#   epoch_on (B, 4)     step index of bar onset, go, movement onset, target
#   labels   per-trial: category, dir_correct, dir_chosen, is_correct,
#                       bar_deg, session_index, split
# The cue is the CORRECT target's direction: the task only exports that
# (DirectionCorrect) plus, on error trials, the foil's direction in
# foil_events_<runTag>.csv. The full three-target layout is not logged, so a
# model that needs it has to be given a layout export from CenterOutTask.m.
# ---------------------------------------------------------------------------
RNN_T_MAX = None            # None = pad to the longest included trial; or an int cap
RNN_EXPORT_PATH = "rnn_dataset.npz"

_ex = RNN_TRIALS[RNN_TRIALS["included"]].reset_index(drop=True)
_T = int(_ex["n_steps"].max()) if RNN_T_MAX is None else int(RNN_T_MAX)
_B = len(_ex)

# length normalisation: the 12-length set mapped to [-1, 1]; boundaries in the
# same units so the model's psychometric function can be read back in deg VA
_all_len = np.sort(RNN_TRIALS["BarSizeVA_deg"].unique())
_len_mid = (_all_len.min() + _all_len.max()) / 2.0
_len_half = (_all_len.max() - _all_len.min()) / 2.0
_norm = lambda deg: (deg - _len_mid) / _len_half
_bounds_deg = globals().get("NOMINAL_BOUNDARIES", [5.05, 6.40])

u = np.zeros((_B, _T, 6), np.float32)
y = np.zeros((_B, _T, 2), np.float32)
xy = np.zeros((_B, _T, 2), np.float32)
valid = np.zeros((_B, _T), bool)
mask_go = np.zeros((_B, _T), bool)
epoch_on = np.full((_B, 4), -1, np.int32)
_k_bar = int(RNN_PRE_STIM_MS / RNN_DT_MS)

for i, row in _ex.iterrows():
    seq = RNN_SEQ[(row["Session"], int(row["Block"]), int(row["TrialNumInBlock"]), int(row["Attempt"]))]
    n = min(seq["grid_ms"].size, _T)
    v = np.nan_to_num(seq["v_cms"][:n]); p = np.nan_to_num(seq["xy_px"][:n])
    y[i, :n] = v; xy[i, :n] = p
    valid[i, :n] = seq["valid"][:n]
    k_go = _k_bar + int(round(row["go_ms"] / RNN_DT_MS))
    k_bar_off = _k_bar + int(round(row.get("bar_dur_ms", 0.0) / RNN_DT_MS)) if np.isfinite(row.get("bar_dur_ms", np.nan)) else k_go
    k_move = _k_bar + int(round(row["move_on_ms"] / RNN_DT_MS)) if np.isfinite(row["move_on_ms"]) else -1
    k_tgt = _k_bar + int(round(row["target_ms"] / RNN_DT_MS)) if np.isfinite(row["target_ms"]) else -1
    u[i, _k_bar:min(k_bar_off, _T), 0] = _norm(row["BarSizeVA_deg"])
    if k_go < _T:
        u[i, k_go:, 1 + int(row["dir_correct_id"])] = 1.0
        u[i, k_go:, 5] = 1.0
        mask_go[i, k_go:] = True
    epoch_on[i] = [_k_bar, k_go, k_move, k_tgt]
# padding and pre-window steps are never "valid", and the mask stops with the data
mask_go &= valid

labels = {
    "category": _ex["category_id"].to_numpy(np.int8),
    "dir_correct": _ex["dir_correct_id"].to_numpy(np.int8),
    "dir_chosen": _ex["dir_chosen_id"].fillna(-1).to_numpy(np.int8),
    "is_correct": _ex["IsCorrect"].to_numpy(np.int8),
    "error_type": _ex["ErrorType"].to_numpy(np.int8) if "ErrorType" in _ex else np.zeros(_B, np.int8),
    "bar_deg": _ex["BarSizeVA_deg"].to_numpy(np.float32),
    "bar_norm": _norm(_ex["BarSizeVA_deg"].to_numpy(np.float32)).astype(np.float32),
    "decision_time_s": _ex["DecisionTime_s"].to_numpy(np.float32),
    "execution_time_s": _ex["ExecutionTime_s"].to_numpy(np.float32),
    "session_index": _ex["SessionIndex"].to_numpy(np.int16),
    "split": _ex["split"].to_numpy(str),
    "block": _ex["Block"].to_numpy(np.int16),
    "trial_in_block": _ex["TrialNumInBlock"].to_numpy(np.int16),
    "attempt": _ex["Attempt"].to_numpy(np.int8),
}
_meta = {
    "dt_ms": RNN_DT_MS, "T": _T, "B": _B, "pre_stim_ms": RNN_PRE_STIM_MS, "post_go_ms": RNN_POST_GO_MS,
    "resampler": RNN_RESAMPLER.name, "cm_per_px": _RNN_CM_PER_PX,
    "u_columns": ["bar_length_norm", "cue_Right_0", "cue_Up_90", "cue_Left_180", "cue_Down_270", "go"],
    "y_columns": ["vx_cms", "vy_cms"], "epoch_on_columns": ["bar", "go", "move", "target"],
    "category_names": list(RNN_CATEGORY_ID), "direction_names": list(DIRECTION_INDEX),
    "length_norm": {"mid_deg": float(_len_mid), "half_range_deg": float(_len_half)},
    "boundaries_deg": [float(b) for b in _bounds_deg],
    "boundaries_norm": [float(_norm(b)) for b in _bounds_deg],
    "hold_noise_cms": RNN_HOLD_NOISE, "sessions": RNN_SESSIONS[["SessionIndex", "Session", "split"]].to_dict("records"),
}
# the join key must be unique or two trials would share one sequence
assert not _ex.duplicated(["Session", "Block", "TrialNumInBlock", "Attempt"]).any(), "duplicate trial key"
assert mask_go.any(axis=1).all(), "some trial has no post-go steps inside the window"

np.savez_compressed(RNN_EXPORT_PATH, u=u, y=y, xy=xy, valid=valid, mask_go=mask_go, epoch_on=epoch_on,
                    meta=json.dumps(_meta, default=float), **{f"label_{k}": v for k, v in labels.items()})
_ex.to_csv("rnn_trials_included.csv", index=False)

print(f"Saved {RNN_EXPORT_PATH}")
for name, arr in (("u", u), ("y", y), ("xy", xy), ("valid", valid), ("mask_go", mask_go), ("epoch_on", epoch_on)):
    print(f"  {name:9s} {arr.shape}  {arr.dtype}")
print(f"  labels    {list(labels)}")
print(f"\nB = {_B} trials, T = {_T} steps ({_T * RNN_DT_MS:.0f} ms), "
      f"valid fraction {valid.mean():.2f}, post-go fraction {mask_go.mean():.2f}.")
print(f"Split sizes: " + ", ".join(f"{s}={int((labels['split'] == s).sum())}" for s in ("train", "val", "test")))
print(f"Bar length -> [-1, 1] with mid {_len_mid:.2f} deg and half-range {_len_half:.2f} deg; "
      f"category boundaries at {np.round(_meta['boundaries_norm'], 3)}.")
print("\nLoad with:\n  d = np.load('rnn_dataset.npz'); meta = json.loads(str(d['meta']))\n"
      "  u, y, valid, mask_go = d['u'], d['y'], d['valid'], d['mask_go']")